In [1]:
import os
import re
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd
import librosa


# ============================================================
# 사용자 설정
# ============================================================

SEGMENTED_AUDIO_ROOT = r"D:\gugak_files_segmented"

OUTPUT_DIR = r"D:\gugak_metadata_labeling"
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "segmented_wav_metadata_autolabeled.csv")
OUTPUT_XLSX = os.path.join(OUTPUT_DIR, "segmented_wav_metadata_autolabeled.xlsx")

SR_CHECK = None  # 원래 sampling rate 유지해서 duration만 확인


# ============================================================
# 1. 키워드 매핑 사전
# ============================================================

INSTRUMENT_KEYWORDS = {
    # 현악기
    "가야금": ("현악기", "가야금"),
    "거문고": ("현악기", "거문고"),
    "해금": ("현악기", "해금"),
    "아쟁": ("현악기", "아쟁"),
    "양금": ("현악기", "양금"),
    "비파": ("현악기", "비파"),

    # 관악기
    "대금": ("관악기", "대금"),
    "소금": ("관악기", "소금"),
    "단소": ("관악기", "단소"),
    "피리": ("관악기", "피리"),
    "태평소": ("관악기", "태평소"),
    "생황": ("관악기", "생황"),
    "훈": ("관악기", "훈"),
    "나각": ("관악기", "나각"),
    "나발": ("관악기", "나발"),

    # 타악기
    "장구": ("타악기", "장구"),
    "북": ("타악기", "북"),
    "좌고": ("타악기", "좌고"),
    "징": ("타악기", "징"),
    "꽹과리": ("타악기", "꽹과리"),
    "바라": ("타악기", "바라"),
    "박": ("타악기", "박"),
    "편종": ("타악기", "편종"),
    "편경": ("타악기", "편경"),
    "운라": ("타악기", "운라"),
    "축": ("타악기", "축"),
    "어": ("타악기", "어"),

    # 성악
    "판소리": ("성악", "voice"),
    "민요": ("성악", "voice"),
    "가곡": ("성악", "voice"),
    "가사": ("성악", "voice"),
    "시조": ("성악", "voice"),
    "범패": ("성악", "voice"),
    "정가": ("성악", "voice"),
    "창": ("성악", "voice"),
}


GENRE_KEYWORDS = {
    "정악": "정악",
    "민속악": "민속악",
    "산조": "산조",
    "판소리": "판소리",
    "민요": "민요",
    "가곡": "가곡",
    "가사": "가사",
    "시조": "시조",
    "정가": "정가",
    "범패": "범패",
    "풍류": "풍류",
    "영산회상": "영산회상",
    "수제천": "수제천",
    "종묘제례악": "종묘제례악",
    "문묘제례악": "문묘제례악",
    "여민락": "여민락",
    "보허자": "보허자",
    "낙양춘": "낙양춘",
    "취타": "취타",
    "대취타": "대취타",
    "농악": "농악",
    "풍물": "풍물",
    "사물놀이": "사물놀이",
    "창작": "창작국악",
    "무속": "무속음악",
    "불교": "불교음악",
}


ENSEMBLE_KEYWORDS = [
    "합주", "관현악", "중주", "병주", "제주", "연주단", "악단", "합창"
]

VOCAL_KEYWORDS = [
    "판소리", "민요", "가곡", "가사", "시조", "정가", "범패", "창", "노래", "소리"
]


# ============================================================
# 2. 유틸 함수
# ============================================================

def normalize_text(text: str) -> str:
    """
    파일명/폴더명 검색을 쉽게 하기 위한 정규화
    """
    text = str(text)
    text = text.replace("_", " ")
    text = text.replace("-", " ")
    text = text.replace(".", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def get_all_wav_files(root_dir: str) -> List[Path]:
    root = Path(root_dir)
    if not root.exists():
        raise FileNotFoundError(f"폴더를 찾을 수 없습니다: {root_dir}")

    wav_files = []
    for path, _, files in os.walk(root):
        for file in files:
            if file.lower().endswith(".wav"):
                wav_files.append(Path(path) / file)

    return sorted(wav_files)


def safe_audio_info(wav_path: Path) -> Dict:
    """
    duration, sampling rate 등을 읽습니다.
    librosa가 실패할 경우 error만 기록합니다.
    """
    try:
        duration = librosa.get_duration(path=str(wav_path))
        y, sr = librosa.load(str(wav_path), sr=SR_CHECK, mono=True, duration=0.1)

        return {
            "audio_read_status": "success",
            "duration_sec": round(float(duration), 3),
            "sampling_rate_detected": int(sr),
            "audio_error": ""
        }

    except Exception as e:
        return {
            "audio_read_status": "error",
            "duration_sec": None,
            "sampling_rate_detected": None,
            "audio_error": str(e)
        }


def infer_instrument(text: str) -> Dict:
    matched = []

    for keyword, (category, instrument) in INSTRUMENT_KEYWORDS.items():
        if keyword in text:
            matched.append({
                "keyword": keyword,
                "instrument_category": category,
                "primary_instrument": instrument
            })

    # 중복 제거
    unique = []
    seen = set()
    for m in matched:
        key = (m["instrument_category"], m["primary_instrument"])
        if key not in seen:
            seen.add(key)
            unique.append(m)

    if len(unique) == 0:
        return {
            "instrument_category": "unknown",
            "primary_instrument": "unknown",
            "instrument_tags": "",
            "instrument_confidence": "low"
        }

    if len(unique) == 1:
        return {
            "instrument_category": unique[0]["instrument_category"],
            "primary_instrument": unique[0]["primary_instrument"],
            "instrument_tags": unique[0]["keyword"],
            "instrument_confidence": "high"
        }

    categories = sorted(set([m["instrument_category"] for m in unique]))
    instruments = sorted(set([m["primary_instrument"] for m in unique]))
    tags = sorted(set([m["keyword"] for m in unique]))

    if len(categories) >= 2:
        instrument_category = "mixed"
    else:
        instrument_category = categories[0]

    return {
        "instrument_category": instrument_category,
        "primary_instrument": ";".join(instruments),
        "instrument_tags": ";".join(tags),
        "instrument_confidence": "medium"
    }


def infer_genre(text: str) -> Dict:
    matched = []

    for keyword, genre in GENRE_KEYWORDS.items():
        if keyword in text:
            matched.append({
                "keyword": keyword,
                "genre_level2": genre
            })

    # 중복 제거
    unique = []
    seen = set()
    for m in matched:
        if m["genre_level2"] not in seen:
            seen.add(m["genre_level2"])
            unique.append(m)

    if len(unique) == 0:
        return {
            "genre_level1": "unknown",
            "genre_level2": "unknown",
            "genre_tags": "",
            "genre_confidence": "low"
        }

    if len(unique) == 1:
        genre2 = unique[0]["genre_level2"]

        if genre2 in ["판소리", "민요", "가곡", "가사", "시조", "정가", "범패"]:
            genre1 = "성악"
        elif genre2 in ["산조", "영산회상", "수제천", "여민락", "보허자", "낙양춘", "취타", "대취타"]:
            genre1 = "기악"
        elif genre2 in ["농악", "풍물", "사물놀이"]:
            genre1 = "타악/연희"
        else:
            genre1 = "unknown"

        return {
            "genre_level1": genre1,
            "genre_level2": genre2,
            "genre_tags": unique[0]["keyword"],
            "genre_confidence": "high"
        }

    genres = sorted(set([m["genre_level2"] for m in unique]))
    tags = sorted(set([m["keyword"] for m in unique]))

    return {
        "genre_level1": "mixed_or_multiple",
        "genre_level2": ";".join(genres),
        "genre_tags": ";".join(tags),
        "genre_confidence": "medium"
    }


def infer_vocal_instrumental_type(text: str, instrument_category: str) -> Dict:
    has_vocal_keyword = any(k in text for k in VOCAL_KEYWORDS)
    has_ensemble_keyword = any(k in text for k in ENSEMBLE_KEYWORDS)

    if has_vocal_keyword and instrument_category in ["현악기", "관악기", "타악기"]:
        vocal_type = "vocal_with_instrument"
        confidence = "medium"
    elif has_vocal_keyword:
        vocal_type = "vocal"
        confidence = "high"
    elif instrument_category in ["현악기", "관악기", "타악기"]:
        vocal_type = "instrumental"
        confidence = "high"
    elif instrument_category == "mixed":
        vocal_type = "mixed_or_ensemble"
        confidence = "medium"
    else:
        vocal_type = "unknown"
        confidence = "low"

    ensemble_flag = 1 if has_ensemble_keyword or instrument_category == "mixed" else 0

    return {
        "vocal_instrumental_type": vocal_type,
        "vocal_instrumental_confidence": confidence,
        "ensemble_flag": ensemble_flag
    }


def infer_all_metadata(row_text: str) -> Dict:
    text = normalize_text(row_text)

    instrument_info = infer_instrument(text)
    genre_info = infer_genre(text)
    vocal_info = infer_vocal_instrumental_type(
        text=text,
        instrument_category=instrument_info["instrument_category"]
    )

    # 수동 검토 플래그
    manual_review_needed = 0
    reasons = []

    if instrument_info["instrument_confidence"] == "low":
        manual_review_needed = 1
        reasons.append("instrument_unknown")

    if genre_info["genre_confidence"] == "low":
        manual_review_needed = 1
        reasons.append("genre_unknown")

    if instrument_info["instrument_confidence"] == "medium":
        reasons.append("multiple_instruments_detected")

    if genre_info["genre_confidence"] == "medium":
        reasons.append("multiple_genres_detected")

    if vocal_info["vocal_instrumental_confidence"] == "low":
        reasons.append("vocal_instrumental_unknown")

    out = {}
    out.update(instrument_info)
    out.update(genre_info)
    out.update(vocal_info)
    out["manual_review_needed"] = manual_review_needed
    out["manual_review_reason"] = ";".join(reasons)

    return out


# ============================================================
# 3. 메인 실행
# ============================================================

def build_autolabeled_metadata(root_dir: str) -> pd.DataFrame:
    root = Path(root_dir)
    wav_files = get_all_wav_files(root_dir)

    rows = []

    for idx, wav_path in enumerate(wav_files, start=1):
        rel_path = wav_path.relative_to(root)
        folder = str(rel_path.parent).replace("\\", "/") if str(rel_path.parent) != "." else ""
        filename = wav_path.name
        stem = wav_path.stem

        # 기존 코드의 segment 파일명 규칙을 고려
        # 예: track_id__seg01.wav
        if "__seg" in stem:
            parent_track_id = stem.split("__seg")[0]
            segment_part = stem.split("__seg")[-1]
            try:
                segment_index_within_track = int(re.sub(r"\D", "", segment_part))
            except Exception:
                segment_index_within_track = None
        else:
            parent_track_id = stem
            segment_index_within_track = None

        text_for_labeling = f"{folder} {filename} {parent_track_id}"

        auto_meta = infer_all_metadata(text_for_labeling)
        audio_info = safe_audio_info(wav_path)

        row = {
            "segment_file_id": f"SEGFILE{idx:06d}",
            "segment_filename": filename,
            "segment_stem": stem,
            "relative_path_segment": str(rel_path).replace("\\", "/"),
            "folder": folder,
            "parent_track_id_inferred": parent_track_id,
            "segment_index_within_track_inferred": segment_index_within_track,
            "absolute_path": str(wav_path),
            "metadata_source_text": normalize_text(text_for_labeling),
        }

        row.update(audio_info)
        row.update(auto_meta)

        rows.append(row)

        if idx % 500 == 0:
            print(f"[PROGRESS] {idx:,} files processed")

    df = pd.DataFrame(rows)

    # 컬럼 순서 정리
    first_cols = [
        "segment_file_id",
        "segment_filename",
        "segment_stem",
        "relative_path_segment",
        "folder",
        "parent_track_id_inferred",
        "segment_index_within_track_inferred",
        "duration_sec",
        "sampling_rate_detected",
        "audio_read_status",
        "audio_error",
        "instrument_category",
        "primary_instrument",
        "instrument_tags",
        "instrument_confidence",
        "genre_level1",
        "genre_level2",
        "genre_tags",
        "genre_confidence",
        "vocal_instrumental_type",
        "vocal_instrumental_confidence",
        "ensemble_flag",
        "manual_review_needed",
        "manual_review_reason",
        "metadata_source_text",
        "absolute_path",
    ]

    existing_first_cols = [c for c in first_cols if c in df.columns]
    other_cols = [c for c in df.columns if c not in existing_first_cols]
    df = df[existing_first_cols + other_cols]

    return df


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("====================================")
    print("Gugak segmented WAV metadata labeling")
    print("====================================")
    print(f"Input folder: {SEGMENTED_AUDIO_ROOT}")

    df = build_autolabeled_metadata(SEGMENTED_AUDIO_ROOT)

    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="autolabeled_metadata", index=False)

        summary = pd.DataFrame({
            "item": [
                "total_wav_files",
                "audio_read_success",
                "audio_read_error",
                "manual_review_needed",
                "manual_review_not_needed",
                "instrument_high_confidence",
                "instrument_medium_confidence",
                "instrument_low_confidence",
                "genre_high_confidence",
                "genre_medium_confidence",
                "genre_low_confidence",
            ],
            "value": [
                len(df),
                int((df["audio_read_status"] == "success").sum()),
                int((df["audio_read_status"] == "error").sum()),
                int((df["manual_review_needed"] == 1).sum()),
                int((df["manual_review_needed"] == 0).sum()),
                int((df["instrument_confidence"] == "high").sum()),
                int((df["instrument_confidence"] == "medium").sum()),
                int((df["instrument_confidence"] == "low").sum()),
                int((df["genre_confidence"] == "high").sum()),
                int((df["genre_confidence"] == "medium").sum()),
                int((df["genre_confidence"] == "low").sum()),
            ]
        })
        summary.to_excel(writer, sheet_name="summary", index=False)

        review_df = df[df["manual_review_needed"] == 1].copy()
        review_df.to_excel(writer, sheet_name="manual_review_needed", index=False)

    print("완료")
    print(f"CSV 저장: {OUTPUT_CSV}")
    print(f"Excel 저장: {OUTPUT_XLSX}")
    print()
    print("총 WAV 파일 수:", len(df))
    print("수동 검토 필요:", int((df["manual_review_needed"] == 1).sum()))
    print("수동 검토 불필요:", int((df["manual_review_needed"] == 0).sum()))


if __name__ == "__main__":
    main()

Gugak segmented WAV metadata labeling
Input folder: D:\gugak_files_segmented
[PROGRESS] 500 files processed
[PROGRESS] 1,000 files processed
[PROGRESS] 1,500 files processed
[PROGRESS] 2,000 files processed
[PROGRESS] 2,500 files processed
[PROGRESS] 3,000 files processed
[PROGRESS] 3,500 files processed
[PROGRESS] 4,000 files processed
[PROGRESS] 4,500 files processed
[PROGRESS] 5,000 files processed
[PROGRESS] 5,500 files processed
[PROGRESS] 6,000 files processed
[PROGRESS] 6,500 files processed
[PROGRESS] 7,000 files processed
[PROGRESS] 7,500 files processed
[PROGRESS] 8,000 files processed
[PROGRESS] 8,500 files processed
완료
CSV 저장: D:\gugak_metadata_labeling\segmented_wav_metadata_autolabeled.csv
Excel 저장: D:\gugak_metadata_labeling\segmented_wav_metadata_autolabeled.xlsx

총 WAV 파일 수: 8928
수동 검토 필요: 3655
수동 검토 불필요: 5273


In [5]:
import pandas as pd
import re
from pathlib import Path

# ============================================================
# 1. 기존 자동 라벨링 결과 불러오기
# ============================================================

xlsx_path = r"D:\gugak_metadata_labeling\segmented_wav_metadata_autolabeled.xlsx"

df = pd.read_excel(xlsx_path, sheet_name="autolabeled_metadata")

print("기존 총 파일 수:", len(df))
print("기존 수동 검토 필요:", int((df["manual_review_needed"] == 1).sum()))


# ============================================================
# 2. 유틸 함수
# ============================================================

def contains_any(text, keywords):
    return any(k in text for k in keywords)


def normalize_source_text(text):
    text = str(text)
    text = text.replace("_", " ").replace("-", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# ============================================================
# 3. 폴더명 기반 보정 함수
# ============================================================

def refine_metadata_by_folder(row):
    row = row.copy()

    folder = str(row.get("folder", ""))
    filename = str(row.get("segment_filename", ""))
    source_text = normalize_source_text(f"{folder} {filename}")

    text = source_text

    # 기존 값
    inst_cat = str(row.get("instrument_category", "unknown"))
    primary_inst = str(row.get("primary_instrument", "unknown"))
    genre1 = str(row.get("genre_level1", "unknown"))
    genre2 = str(row.get("genre_level2", "unknown"))
    vocal_type = str(row.get("vocal_instrumental_type", "unknown"))

    applied_rules = []

    # ------------------------------------------------------------
    # A. 해금창사 예외 규칙
    # 중요: 해금창사는 성악/창사가 아니라 해금 독주로 처리
    # ------------------------------------------------------------
    if "해금창사" in text:
        inst_cat = "현악기"
        primary_inst = "해금"
        vocal_type = "instrumental"
        genre1 = "기악"
        genre2 = "해금독주"
        applied_rules.append("haegeum_changsa_as_haegeum_solo")

    # ------------------------------------------------------------
    # B. 일반 여창/남창/창사 계열
    # 단, 해금창사는 위에서 이미 처리했으므로 제외
    # ------------------------------------------------------------
    elif contains_any(text, ["창사", "여창", "남창"]):
        genre1 = "성악"
        genre2 = "창사"

        if "여창" in text:
            vocal_type = "female_vocal"
            primary_inst = "voice"
            inst_cat = "성악"
            applied_rules.append("female_vocal_changsa")

        elif "남창" in text:
            vocal_type = "male_vocal"
            primary_inst = "voice"
            inst_cat = "성악"
            applied_rules.append("male_vocal_changsa")

        else:
            vocal_type = "vocal"
            if inst_cat == "unknown":
                inst_cat = "성악"
            if primary_inst == "unknown":
                primary_inst = "voice"
            applied_rules.append("changsa_vocal")

    # ------------------------------------------------------------
    # C. 궁중정재 / 정재 무용 관련 곡명
    # 해금창사인 경우에도 궁중정재 곡명은 보조정보로만 기록
    # ------------------------------------------------------------
    court_dance_keywords = [
        "연백복지무", "사선무", "연화무", "연화대무", "포구락",
        "수연장", "근천정", "제수창", "보상무", "최화무",
        "몽금척", "오양선", "춘앵전", "첩승무", "아박무",
        "무신향", "춘광호", "봉래의", "곡파", "수명명",
        "수악절", "어부사"
    ]

    if contains_any(text, court_dance_keywords):
        row["related_repertoire_inferred"] = "궁중정재"
        applied_rules.append("court_dance_repertoire_detected")

        # 해금창사가 아닌 경우에만 genre를 궁중정재 쪽으로 보정
        if "haegeum_changsa_as_haegeum_solo" not in applied_rules:
            if genre1 == "unknown" or genre1 == "성악":
                genre1 = "궁중정재"

            if genre2 == "unknown" or genre2 == "창사":
                if contains_any(text, ["창사", "여창", "남창"]):
                    genre2 = "궁중정재_창사"
                else:
                    genre2 = "궁중정재"
    else:
        row["related_repertoire_inferred"] = row.get("related_repertoire_inferred", "")

    # ------------------------------------------------------------
    # D. 연례악
    # ------------------------------------------------------------
    if "연례악" in text:
        genre1 = "정악"
        genre2 = "연례악"
        applied_rules.append("yeonlyeak")

        if "대금" in text:
            inst_cat = "관악기"
            primary_inst = "대금"
            vocal_type = "instrumental"
            applied_rules.append("daegeum_yeonlyeak")

    # ------------------------------------------------------------
    # E. 사물 / 사물놀이 / 꽹과리 / 장구
    # ------------------------------------------------------------
    if "사물" in text:
        genre1 = "타악/연희"
        genre2 = "사물놀이"
        inst_cat = "타악기"
        vocal_type = "instrumental"
        applied_rules.append("samul")

        instruments = []
        if "꽹과리" in text:
            instruments.append("꽹과리")
        if "장구" in text:
            instruments.append("장구")
        if "북" in text:
            instruments.append("북")
        if "징" in text:
            instruments.append("징")

        if instruments:
            primary_inst = ";".join(sorted(set(instruments)))
        elif primary_inst == "unknown":
            primary_inst = "percussion_ensemble"

    # ------------------------------------------------------------
    # F. 푸너리 / 동해안 / 무속 타악 계열
    # ------------------------------------------------------------
    if contains_any(text, ["푸너리", "드렁갱이", "삼오장", "동해안"]):
        genre1 = "무속음악"
        genre2 = "푸너리"
        inst_cat = "타악기"
        vocal_type = "instrumental"
        applied_rules.append("punori_shamanic_percussion")

        instruments = []
        if "징" in text:
            instruments.append("징")
        if "바라" in text:
            instruments.append("바라")
        if "장구" in text:
            instruments.append("장구")
        if "꽹과리" in text:
            instruments.append("꽹과리")

        if instruments:
            primary_inst = ";".join(sorted(set(instruments)))
        elif primary_inst == "unknown":
            primary_inst = "percussion"

        if "동해안" in text:
            genre2 = "동해안_푸너리"
            applied_rules.append("donghaean_punori")

    # ------------------------------------------------------------
    # G. 장단명 보조 규칙
    # ------------------------------------------------------------
    jangdan_keywords = [
        "일채", "이채", "삼채", "자진삼채", "굿거리",
        "자진모리", "중모리", "중중모리", "휘모리", "엇모리",
        "삼오장"
    ]

    matched_jangdan = [k for k in jangdan_keywords if k in text]

    if matched_jangdan:
        row["jangdan_inferred"] = ";".join(matched_jangdan)
        applied_rules.append("jangdan_detected")
    else:
        row["jangdan_inferred"] = row.get("jangdan_inferred", "")

    # ------------------------------------------------------------
    # H. 악기명 직접 보정
    # 단, 해금창사는 위에서 이미 해금독주로 처리됨
    # ------------------------------------------------------------
    if "대금" in text:
        if inst_cat == "unknown":
            inst_cat = "관악기"
        if primary_inst == "unknown":
            primary_inst = "대금"
        if vocal_type == "unknown":
            vocal_type = "instrumental"
        applied_rules.append("daegeum")

    if "해금" in text:
        if inst_cat == "unknown":
            inst_cat = "현악기"
        if primary_inst == "unknown":
            primary_inst = "해금"
        if vocal_type == "unknown":
            vocal_type = "instrumental"
        applied_rules.append("haegeum")

    if "장구" in text:
        if inst_cat == "unknown":
            inst_cat = "타악기"
        if primary_inst == "unknown":
            primary_inst = "장구"
        if vocal_type == "unknown":
            vocal_type = "instrumental"
        applied_rules.append("janggu")

    if "꽹과리" in text:
        if inst_cat == "unknown":
            inst_cat = "타악기"
        if primary_inst == "unknown":
            primary_inst = "꽹과리"
        if vocal_type == "unknown":
            vocal_type = "instrumental"
        applied_rules.append("kkwaenggwari")

    if "징" in text:
        if inst_cat == "unknown":
            inst_cat = "타악기"
        if primary_inst == "unknown":
            primary_inst = "징"
        if vocal_type == "unknown":
            vocal_type = "instrumental"
        applied_rules.append("jing")

    if "바라" in text:
        if inst_cat == "unknown":
            inst_cat = "타악기"
        if primary_inst == "unknown":
            primary_inst = "바라"
        if vocal_type == "unknown":
            vocal_type = "instrumental"
        applied_rules.append("bara")

    # ------------------------------------------------------------
    # I. 최종 보정값 저장
    # ------------------------------------------------------------
    row["instrument_category_refined"] = inst_cat
    row["primary_instrument_refined"] = primary_inst
    row["genre_level1_refined"] = genre1
    row["genre_level2_refined"] = genre2
    row["vocal_instrumental_type_refined"] = vocal_type
    row["folder_refinement_rules"] = ";".join(sorted(set(applied_rules)))

    if row["folder_refinement_rules"]:
        row["metadata_refinement_status"] = "refined_by_folder_rule"
    else:
        row["metadata_refinement_status"] = "not_refined"

    return row


# ============================================================
# 4. 보정 적용
# ============================================================

df_refined = df.apply(refine_metadata_by_folder, axis=1)


# ============================================================
# 5. 수동 검토 필요 여부 재계산
# ============================================================

def recompute_manual_review(row):
    reasons = []

    if str(row.get("instrument_category_refined", "unknown")) in ["unknown", "", "nan"]:
        reasons.append("instrument_unknown")

    if str(row.get("primary_instrument_refined", "unknown")) in ["unknown", "", "nan"]:
        reasons.append("primary_instrument_unknown")

    if str(row.get("genre_level1_refined", "unknown")) in ["unknown", "", "nan"]:
        reasons.append("genre_level1_unknown")

    if str(row.get("genre_level2_refined", "unknown")) in ["unknown", "", "nan"]:
        reasons.append("genre_level2_unknown")

    if str(row.get("vocal_instrumental_type_refined", "unknown")) in ["unknown", "", "nan"]:
        reasons.append("vocal_instrumental_unknown")

    row["manual_review_needed_refined"] = 1 if len(reasons) > 0 else 0
    row["manual_review_reason_refined"] = ";".join(reasons)

    return row


df_refined = df_refined.apply(recompute_manual_review, axis=1)


# ============================================================
# 6. 해금창사 처리 확인용 테이블
# ============================================================

haegeum_changsa_check = df_refined[
    df_refined["folder"].astype(str).str.contains("해금창사", na=False)
].copy()

print("해금창사 파일 수:", len(haegeum_changsa_check))

if len(haegeum_changsa_check) > 0:
    print(
        haegeum_changsa_check[
            [
                "folder",
                "segment_filename",
                "instrument_category_refined",
                "primary_instrument_refined",
                "genre_level1_refined",
                "genre_level2_refined",
                "vocal_instrumental_type_refined",
                "folder_refinement_rules"
            ]
        ].head(20)
    )


# ============================================================
# 7. 저장
# ============================================================

output_path = r"D:\gugak_metadata_labeling\segmented_wav_metadata_refined_v3_haegeum_fixed.xlsx"
output_csv = r"D:\gugak_metadata_labeling\segmented_wav_metadata_refined_v3_haegeum_fixed.csv"

df_refined.to_csv(output_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df_refined.to_excel(writer, sheet_name="refined_metadata", index=False)

    summary = pd.DataFrame({
        "item": [
            "total_wav_files",
            "manual_review_needed_original",
            "manual_review_needed_refined",
            "manual_review_reduced_count",
            "manual_review_reduced_percent",
            "refined_by_folder_rule",
            "not_refined",
            "haegeum_changsa_files"
        ],
        "value": [
            len(df_refined),
            int((df_refined["manual_review_needed"] == 1).sum()),
            int((df_refined["manual_review_needed_refined"] == 1).sum()),
            int((df_refined["manual_review_needed"] == 1).sum() - (df_refined["manual_review_needed_refined"] == 1).sum()),
            round(
                100 * (
                    int((df_refined["manual_review_needed"] == 1).sum()) -
                    int((df_refined["manual_review_needed_refined"] == 1).sum())
                ) / max(int((df_refined["manual_review_needed"] == 1).sum()), 1),
                2
            ),
            int((df_refined["metadata_refinement_status"] == "refined_by_folder_rule").sum()),
            int((df_refined["metadata_refinement_status"] == "not_refined").sum()),
            len(haegeum_changsa_check)
        ]
    })

    summary.to_excel(writer, sheet_name="summary", index=False)

    still_review = df_refined[df_refined["manual_review_needed_refined"] == 1].copy()
    still_review.to_excel(writer, sheet_name="manual_review_still_needed", index=False)

    folder_counts_after = (
        still_review["folder"]
        .fillna("")
        .value_counts()
        .reset_index()
    )
    folder_counts_after.columns = ["folder", "n"]
    folder_counts_after.to_excel(writer, sheet_name="folder_counts_after", index=False)

    haegeum_changsa_check.to_excel(writer, sheet_name="haegeum_changsa_check", index=False)


print("저장 완료:", output_path)
print("CSV 저장 완료:", output_csv)
print()
print("기존 수동 검토 필요:", int((df_refined["manual_review_needed"] == 1).sum()))
print("보정 후 수동 검토 필요:", int((df_refined["manual_review_needed_refined"] == 1).sum()))
print("감소 개수:", int((df_refined["manual_review_needed"] == 1).sum() - (df_refined["manual_review_needed_refined"] == 1).sum()))
print("해금창사 파일 수:", len(haegeum_changsa_check))

기존 총 파일 수: 8928
기존 수동 검토 필요: 3655
해금창사 파일 수: 299
                     folder                              segment_filename  \
8615  해금창사수악절창사152-연백복지무213  해금창사수악절창사152-연백복지무213__S3-071-152__seg01.wav   
8616  해금창사수악절창사152-연백복지무213  해금창사수악절창사152-연백복지무213__S3-071-153__seg01.wav   
8617  해금창사수악절창사152-연백복지무213  해금창사수악절창사152-연백복지무213__S3-071-155__seg01.wav   
8618  해금창사수악절창사152-연백복지무213  해금창사수악절창사152-연백복지무213__S3-071-156__seg01.wav   
8619  해금창사수악절창사152-연백복지무213  해금창사수악절창사152-연백복지무213__S3-071-157__seg01.wav   
8620  해금창사수악절창사152-연백복지무213  해금창사수악절창사152-연백복지무213__S3-071-158__seg01.wav   
8621  해금창사수악절창사152-연백복지무213  해금창사수악절창사152-연백복지무213__S3-071-159__seg01.wav   
8622  해금창사수악절창사152-연백복지무213  해금창사수악절창사152-연백복지무213__S3-071-160__seg01.wav   
8623  해금창사수악절창사152-연백복지무213  해금창사수악절창사152-연백복지무213__S3-071-161__seg01.wav   
8624  해금창사수악절창사152-연백복지무213  해금창사수악절창사152-연백복지무213__S3-072-194__seg01.wav   
8625  해금창사수악절창사152-연백복지무213  해금창사수악절창사152-연백복지무213__S3-072-194__seg02.wav   
8626  해금창사수악절창사152-연백복지무213

In [7]:
import pandas as pd
import re

# ============================================================
# 1. 기존 refined v3 파일 불러오기
# ============================================================

input_xlsx = r"D:\gugak_metadata_labeling\segmented_wav_metadata_refined_v3_haegeum_fixed.xlsx"

df = pd.read_excel(input_xlsx, sheet_name="refined_metadata")

print("기존 총 파일 수:", len(df))
print("기존 보정 후 수동 검토 필요:", int((df["manual_review_needed_refined"] == 1).sum()))


# ============================================================
# 2. 풍류 계열 추가 보정 함수
# ============================================================

def normalize_text(x):
    x = str(x)
    x = x.replace("_", " ").replace("-", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()


def refine_pungnyu_rules(row):
    row = row.copy()

    folder = str(row.get("folder", ""))
    filename = str(row.get("segment_filename", ""))
    source_text = normalize_text(f"{folder} {filename}")

    genre1 = str(row.get("genre_level1_refined", "unknown"))
    genre2 = str(row.get("genre_level2_refined", "unknown"))
    rules = str(row.get("folder_refinement_rules", ""))

    added_rules = []

    # ------------------------------------------------------------
    # A. 민간풍류 / 경기대풍류 / 대풍류
    # ------------------------------------------------------------
    if "민간풍류" in source_text or "경기대풍류" in source_text or "대풍류" in source_text:
        genre1 = "민속악"
        genre2 = "대풍류" if "대풍류" in source_text else "민간풍류"
        added_rules.append("pungnyu_minsokak_daepungnyu")

    # ------------------------------------------------------------
    # B. 거문고풍류 / 단소풍류 / 일반 풍류
    # ------------------------------------------------------------
    elif "풍류" in source_text or genre2 == "풍류":
        genre1 = "정악"

        if "거문고풍류" in source_text:
            genre2 = "거문고풍류"
            added_rules.append("geomungo_pungnyu_jeongak")
        elif "단소풍류" in source_text:
            genre2 = "단소풍류"
            added_rules.append("danso_pungnyu_jeongak")
        elif "줄풍류" in source_text:
            genre2 = "줄풍류"
            added_rules.append("jul_pungnyu_jeongak")
        else:
            genre2 = "풍류"
            added_rules.append("generic_pungnyu_jeongak")

    # ------------------------------------------------------------
    # C. 보정 결과 저장
    # ------------------------------------------------------------
    row["genre_level1_refined"] = genre1
    row["genre_level2_refined"] = genre2

    if added_rules:
        if rules and rules != "nan":
            row["folder_refinement_rules"] = rules + ";" + ";".join(added_rules)
        else:
            row["folder_refinement_rules"] = ";".join(added_rules)

        row["metadata_refinement_status"] = "refined_by_additional_pungnyu_rule"

    return row


df2 = df.apply(refine_pungnyu_rules, axis=1)


# ============================================================
# 3. 수동 검토 필요 여부 재계산
# ============================================================

def recompute_manual_review(row):
    reasons = []

    if str(row.get("instrument_category_refined", "unknown")) in ["unknown", "", "nan"]:
        reasons.append("instrument_unknown")

    if str(row.get("primary_instrument_refined", "unknown")) in ["unknown", "", "nan"]:
        reasons.append("primary_instrument_unknown")

    if str(row.get("genre_level1_refined", "unknown")) in ["unknown", "", "nan"]:
        reasons.append("genre_level1_unknown")

    if str(row.get("genre_level2_refined", "unknown")) in ["unknown", "", "nan"]:
        reasons.append("genre_level2_unknown")

    if str(row.get("vocal_instrumental_type_refined", "unknown")) in ["unknown", "", "nan"]:
        reasons.append("vocal_instrumental_unknown")

    row["manual_review_needed_refined"] = 1 if len(reasons) > 0 else 0
    row["manual_review_reason_refined"] = ";".join(reasons)

    return row


df2 = df2.apply(recompute_manual_review, axis=1)


# ============================================================
# 4. 풍류 보정 확인용 테이블
# ============================================================

pungnyu_check = df2[
    df2["folder"].astype(str).str.contains("풍류", na=False)
    | df2["segment_filename"].astype(str).str.contains("풍류", na=False)
].copy()

print("풍류 관련 파일 수:", len(pungnyu_check))

check_cols = [
    "folder",
    "segment_filename",
    "instrument_category_refined",
    "primary_instrument_refined",
    "genre_level1_refined",
    "genre_level2_refined",
    "vocal_instrumental_type_refined",
    "manual_review_needed_refined",
    "manual_review_reason_refined",
    "folder_refinement_rules"
]

print(pungnyu_check[check_cols].head(50))


# ============================================================
# 5. 저장
# ============================================================

output_xlsx = r"D:\gugak_metadata_labeling\segmented_wav_metadata_refined_v4_pungnyu_fixed.xlsx"
output_csv = r"D:\gugak_metadata_labeling\segmented_wav_metadata_refined_v4_pungnyu_fixed.csv"

df2.to_csv(output_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
    df2.to_excel(writer, sheet_name="refined_metadata", index=False)

    summary = pd.DataFrame({
        "item": [
            "total_wav_files",
            "manual_review_needed_before_pungnyu_fix",
            "manual_review_needed_after_pungnyu_fix",
            "manual_review_reduced_by_pungnyu_fix",
            "final_auto_or_rule_based_labeled",
            "final_manual_review_rate_percent",
            "pungnyu_related_files"
        ],
        "value": [
            len(df),
            int((df["manual_review_needed_refined"] == 1).sum()),
            int((df2["manual_review_needed_refined"] == 1).sum()),
            int((df["manual_review_needed_refined"] == 1).sum() - (df2["manual_review_needed_refined"] == 1).sum()),
            int((df2["manual_review_needed_refined"] == 0).sum()),
            round(int((df2["manual_review_needed_refined"] == 1).sum()) / len(df2) * 100, 2),
            len(pungnyu_check)
        ]
    })

    summary.to_excel(writer, sheet_name="summary", index=False)

    still_review = df2[df2["manual_review_needed_refined"] == 1].copy()
    still_review.to_excel(writer, sheet_name="manual_review_still_needed", index=False)

    folder_counts_after = (
        still_review["folder"]
        .fillna("")
        .value_counts()
        .reset_index()
    )
    folder_counts_after.columns = ["folder", "n"]
    folder_counts_after.to_excel(writer, sheet_name="folder_counts_after", index=False)

    pungnyu_check.to_excel(writer, sheet_name="pungnyu_check", index=False)

print("저장 완료:", output_xlsx)
print("CSV 저장 완료:", output_csv)
print()
print("풍류 보정 전 수동 검토 필요:", int((df["manual_review_needed_refined"] == 1).sum()))
print("풍류 보정 후 수동 검토 필요:", int((df2["manual_review_needed_refined"] == 1).sum()))
print("감소 개수:", int((df["manual_review_needed_refined"] == 1).sum() - (df2["manual_review_needed_refined"] == 1).sum()))

기존 총 파일 수: 8928
기존 보정 후 수동 검토 필요: 469
풍류 관련 파일 수: 151
                   folder                            segment_filename  \
161           거문고풍류상영산-군악         거문고풍류상영산-군악__s2-111-010g__seg01.wav   
162           거문고풍류상영산-군악         거문고풍류상영산-군악__s2-111-010g__seg02.wav   
163           거문고풍류상영산-군악         거문고풍류상영산-군악__s2-111-010g__seg03.wav   
164           거문고풍류상영산-군악         거문고풍류상영산-군악__s2-111-020g__seg01.wav   
165           거문고풍류상영산-군악         거문고풍류상영산-군악__s2-111-020g__seg02.wav   
166           거문고풍류상영산-군악         거문고풍류상영산-군악__s2-111-020g__seg03.wav   
167           거문고풍류상영산-군악         거문고풍류상영산-군악__s2-113-010g__seg01.wav   
168           거문고풍류상영산-군악         거문고풍류상영산-군악__s2-113-020g__seg01.wav   
169           거문고풍류상영산-군악         거문고풍류상영산-군악__s2-113-030g__seg01.wav   
170           거문고풍류상영산-군악         거문고풍류상영산-군악__s2-116-010g__seg01.wav   
171           거문고풍류상영산-군악         거문고풍류상영산-군악__s2-116-020g__seg01.wav   
172           거문고풍류상영산-군악         거문고풍류상영산-군악__s2-116-030g__seg01.wav 

In [9]:
import pandas as pd
import re

# ============================================================
# 1. v4 파일 불러오기
# ============================================================

input_xlsx = r"D:\gugak_metadata_labeling\segmented_wav_metadata_refined_v4_pungnyu_fixed.xlsx"

df = pd.read_excel(input_xlsx, sheet_name="refined_metadata")

print("기존 총 파일 수:", len(df))
print("v4 수동 검토 필요:", int((df["manual_review_needed_refined"] == 1).sum()))


# ============================================================
# 2. 유틸
# ============================================================

def normalize_text(x):
    x = str(x)
    x = x.replace("_", " ").replace("-", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()


def append_rule(existing, new_rule):
    existing = "" if pd.isna(existing) else str(existing)
    rules = [] if existing in ["", "nan"] else existing.split(";")
    rules.append(new_rule)
    return ";".join(sorted(set([r for r in rules if r])))


# ============================================================
# 3. 남은 319개 패턴 보정 함수
# ============================================================

def refine_remaining_319_rules(row):
    row = row.copy()

    folder = str(row.get("folder", ""))
    filename = str(row.get("segment_filename", ""))
    text = normalize_text(f"{folder} {filename}")

    inst_cat = str(row.get("instrument_category_refined", "unknown"))
    primary_inst = str(row.get("primary_instrument_refined", "unknown"))
    genre1 = str(row.get("genre_level1_refined", "unknown"))
    genre2 = str(row.get("genre_level2_refined", "unknown"))
    vocal_type = str(row.get("vocal_instrumental_type_refined", "unknown"))
    rules = row.get("folder_refinement_rules", "")

    # ------------------------------------------------------------
    # A. 정악장구 계열
    # 예: 정악장구상사별곡-양양가, 정악장구백구사-길군악,
    #     정악장구황계사-매화가, 정악장구수양산가-처용가
    # ------------------------------------------------------------
    if "정악장구" in text:
        inst_cat = "타악기"
        primary_inst = "장구"
        genre1 = "정악"
        genre2 = "정악장구"
        vocal_type = "instrumental"
        rules = append_rule(rules, "jeongak_janggu")
        
        # 세부 레퍼토리 보조 태그
        repertoire_terms = [
            "상사별곡", "양양가", "백구사", "길군악",
            "황계사", "매화가", "수양산가", "처용가"
        ]
        matched = [k for k in repertoire_terms if k in text]
        row["related_repertoire_inferred"] = ";".join(matched) if matched else row.get("related_repertoire_inferred", "")

    # ------------------------------------------------------------
    # B. 반주장구 계열
    # ------------------------------------------------------------
    if "반주장구정악" in text:
        inst_cat = "타악기"
        primary_inst = "장구"
        genre1 = "정악"
        genre2 = "반주장구"
        vocal_type = "instrumental"
        rules = append_rule(rules, "banju_janggu_jeongak")

    if "반주장구민속악" in text:
        inst_cat = "타악기"
        primary_inst = "장구"
        genre1 = "민속악"
        genre2 = "반주장구"
        vocal_type = "instrumental"
        rules = append_rule(rules, "banju_janggu_minsokak")

    # ------------------------------------------------------------
    # C. 아쟁남도잡가
    # ------------------------------------------------------------
    if "아쟁남도잡가" in text:
        inst_cat = "현악기"
        primary_inst = "아쟁"
        genre1 = "민속악"
        genre2 = "남도잡가"
        vocal_type = "instrumental"
        rules = append_rule(rules, "ajaeng_namdo_japga")

    # ------------------------------------------------------------
    # D. 태평소 독주/행악 계열
    # ------------------------------------------------------------
    if "태평소독주시나위" in text:
        inst_cat = "관악기"
        primary_inst = "태평소"
        genre1 = "민속악"
        genre2 = "시나위"
        vocal_type = "instrumental"
        rules = append_rule(rules, "taepyeongso_sinawi")

    if "태평소독주능게" in text:
        inst_cat = "관악기"
        primary_inst = "태평소"
        genre1 = "민속악"
        genre2 = "능게"
        vocal_type = "instrumental"
        rules = append_rule(rules, "taepyeongso_neungge")

    if "태평소독주메나리" in text:
        inst_cat = "관악기"
        primary_inst = "태평소"
        genre1 = "민속악"
        genre2 = "메나리"
        vocal_type = "instrumental"
        rules = append_rule(rules, "taepyeongso_menari")

    if "태평소행악" in text:
        inst_cat = "관악기"
        primary_inst = "태평소"
        genre1 = "행악"
        genre2 = "행악"
        vocal_type = "instrumental"
        rules = append_rule(rules, "taepyeongso_haengak")

    # 장단 보조 태그
    jangdan_terms = ["굿거리", "자진모리", "휘모리"]
    matched_jangdan = [k for k in jangdan_terms if k in text]
    if matched_jangdan:
        existing_jangdan = "" if pd.isna(row.get("jangdan_inferred", "")) else str(row.get("jangdan_inferred", ""))
        combined = sorted(set([x for x in existing_jangdan.split(";") if x] + matched_jangdan))
        row["jangdan_inferred"] = ";".join(combined)
        rules = append_rule(rules, "jangdan_detected_remaining")

    # ------------------------------------------------------------
    # E. 징바라 굿/가락/웃다리 계열
    # ------------------------------------------------------------
    if "징바라경기도당굿" in text:
        inst_cat = "타악기"
        primary_inst = "바라;징"
        genre1 = "무속음악"
        genre2 = "경기도당굿"
        vocal_type = "instrumental"
        rules = append_rule(rules, "jing_bara_gyeonggi_dodanggut")

    if "징바라진도씻김굿" in text:
        inst_cat = "타악기"
        primary_inst = "바라;징"
        genre1 = "무속음악"
        genre2 = "진도씻김굿"
        vocal_type = "instrumental"
        rules = append_rule(rules, "jing_bara_jindo_sitkimgut")

    if "징바라영남가락" in text:
        inst_cat = "타악기"
        primary_inst = "바라;징"
        genre1 = "민속악"
        genre2 = "영남가락"
        vocal_type = "instrumental"
        rules = append_rule(rules, "jing_bara_yeongnam_garak")

    if "징바라웃다리" in text:
        inst_cat = "타악기"
        primary_inst = "바라;징"
        genre1 = "농악"
        genre2 = "웃다리"
        vocal_type = "instrumental"
        rules = append_rule(rules, "jing_bara_utdari")

    # ------------------------------------------------------------
    # F. 무악 계열
    # ------------------------------------------------------------
    if "진도장구무악" in text:
        inst_cat = "타악기"
        primary_inst = "장구"
        genre1 = "무속음악"
        genre2 = "진도무악"
        vocal_type = "instrumental"
        rules = append_rule(rules, "jindo_janggu_muak")

    if "북무악" in text:
        inst_cat = "타악기"
        primary_inst = "북"
        genre1 = "무속음악"
        genre2 = "무악"
        vocal_type = "instrumental"
        rules = append_rule(rules, "buk_muak")

    # ------------------------------------------------------------
    # G. 소리북
    # ------------------------------------------------------------
    if "소리북" in text:
        inst_cat = "타악기"
        primary_inst = "소리북"
        genre1 = "성악반주"
        genre2 = "소리북"
        vocal_type = "instrumental_accompaniment"
        rules = append_rule(rules, "soribuk_accompaniment")

    # ------------------------------------------------------------
    # H. 대피리 연례
    # ------------------------------------------------------------
    if "대피리연례" in text:
        inst_cat = "관악기"
        primary_inst = "대피리"
        genre1 = "정악"
        genre2 = "연례악"
        vocal_type = "instrumental"
        rules = append_rule(rules, "daepiri_yeonlyeak")

    # ------------------------------------------------------------
    # I. 최종 저장
    # ------------------------------------------------------------
    row["instrument_category_refined"] = inst_cat
    row["primary_instrument_refined"] = primary_inst
    row["genre_level1_refined"] = genre1
    row["genre_level2_refined"] = genre2
    row["vocal_instrumental_type_refined"] = vocal_type
    row["folder_refinement_rules"] = rules

    if rules and rules != "nan":
        row["metadata_refinement_status"] = "refined_by_remaining_319_rule"

    return row


# ============================================================
# 4. 적용
# ============================================================

df2 = df.apply(refine_remaining_319_rules, axis=1)


# ============================================================
# 5. manual review 재계산
# ============================================================

def recompute_manual_review(row):
    reasons = []

    if str(row.get("instrument_category_refined", "unknown")) in ["unknown", "", "nan"]:
        reasons.append("instrument_unknown")

    if str(row.get("primary_instrument_refined", "unknown")) in ["unknown", "", "nan"]:
        reasons.append("primary_instrument_unknown")

    if str(row.get("genre_level1_refined", "unknown")) in ["unknown", "", "nan"]:
        reasons.append("genre_level1_unknown")

    if str(row.get("genre_level2_refined", "unknown")) in ["unknown", "", "nan"]:
        reasons.append("genre_level2_unknown")

    if str(row.get("vocal_instrumental_type_refined", "unknown")) in ["unknown", "", "nan"]:
        reasons.append("vocal_instrumental_unknown")

    row["manual_review_needed_refined"] = 1 if len(reasons) > 0 else 0
    row["manual_review_reason_refined"] = ";".join(reasons)

    return row


df2 = df2.apply(recompute_manual_review, axis=1)


# ============================================================
# 6. 결과 확인
# ============================================================

before = int((df["manual_review_needed_refined"] == 1).sum())
after = int((df2["manual_review_needed_refined"] == 1).sum())

print("v4 수동 검토 필요:", before)
print("v5 보정 후 수동 검토 필요:", after)
print("감소 개수:", before - after)
print("최종 미해결 비율(%):", round(after / len(df2) * 100, 2))


# ============================================================
# 7. 저장
# ============================================================

output_xlsx = r"D:\gugak_metadata_labeling\segmented_wav_metadata_refined_v5_remaining_fixed.xlsx"
output_csv = r"D:\gugak_metadata_labeling\segmented_wav_metadata_refined_v5_remaining_fixed.csv"

df2.to_csv(output_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
    df2.to_excel(writer, sheet_name="refined_metadata", index=False)

    summary = pd.DataFrame({
        "item": [
            "total_wav_files",
            "manual_review_needed_v4",
            "manual_review_needed_v5",
            "manual_review_reduced_by_v5",
            "final_auto_or_rule_based_labeled",
            "final_manual_review_rate_percent"
        ],
        "value": [
            len(df2),
            before,
            after,
            before - after,
            int((df2["manual_review_needed_refined"] == 0).sum()),
            round(after / len(df2) * 100, 2)
        ]
    })

    summary.to_excel(writer, sheet_name="summary", index=False)

    still_review = df2[df2["manual_review_needed_refined"] == 1].copy()
    still_review.to_excel(writer, sheet_name="manual_review_still_needed", index=False)

    folder_counts_after = (
        still_review["folder"]
        .fillna("")
        .value_counts()
        .reset_index()
    )
    folder_counts_after.columns = ["folder", "n"]
    folder_counts_after.to_excel(writer, sheet_name="folder_counts_after", index=False)

print("저장 완료:", output_xlsx)
print("CSV 저장 완료:", output_csv)

기존 총 파일 수: 8928
v4 수동 검토 필요: 319
v4 수동 검토 필요: 319
v5 보정 후 수동 검토 필요: 0
감소 개수: 319
최종 미해결 비율(%): 0.0
저장 완료: D:\gugak_metadata_labeling\segmented_wav_metadata_refined_v5_remaining_fixed.xlsx
CSV 저장 완료: D:\gugak_metadata_labeling\segmented_wav_metadata_refined_v5_remaining_fixed.csv


In [10]:
import pandas as pd

summary = pd.DataFrame({
    "item": [
        "total_segment_wav_files",
        "initial_manual_review_needed",
        "after_major_korean_music_rule_refinement",
        "after_pungnyu_refinement",
        "after_remaining_category_refinement",
        "final_unresolved_files",
        "final_unresolved_rate_percent",
        "final_rule_based_labeled_files",
        "final_rule_based_labeled_rate_percent"
    ],
    "value": [
        8928,
        3655,
        469,
        319,
        0,
        0,
        0.0,
        8928,
        100.0
    ]
})

summary.to_csv(
    r"D:\gugak_metadata_labeling\metadata_labeling_validation_summary_final.csv",
    index=False,
    encoding="utf-8-sig"
)

summary.to_excel(
    r"D:\gugak_metadata_labeling\metadata_labeling_validation_summary_final.xlsx",
    index=False
)

print(summary)

                                       item   value
0                   total_segment_wav_files  8928.0
1              initial_manual_review_needed  3655.0
2  after_major_korean_music_rule_refinement   469.0
3                  after_pungnyu_refinement   319.0
4       after_remaining_category_refinement     0.0
5                    final_unresolved_files     0.0
6             final_unresolved_rate_percent     0.0
7            final_rule_based_labeled_files  8928.0
8     final_rule_based_labeled_rate_percent   100.0


In [14]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa

warnings.filterwarnings("ignore")

SEGMENTED_AUDIO_ROOT = r"D:\gugak_files_segmented"

OUTPUT_DIR = r"D:\gugak_scientific_data_package_v1"
DATA_DIR = os.path.join(OUTPUT_DIR, "data")
VALIDATION_DIR = os.path.join(OUTPUT_DIR, "validation")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(VALIDATION_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(DATA_DIR, "segment_acoustic_features_v2_fast.csv")
OUTPUT_XLSX = os.path.join(DATA_DIR, "segment_acoustic_features_v2_fast.xlsx")
ERROR_CSV = os.path.join(VALIDATION_DIR, "feature_extraction_errors_v2_fast.csv")

SR = 22050
N_MFCC = 13


def get_all_wav_files(root_dir):
    root = Path(root_dir)
    wav_files = []
    for path, _, files in os.walk(root):
        for file in files:
            if file.lower().endswith(".wav"):
                wav_files.append(Path(path) / file)
    return sorted(wav_files)


def safe_mean(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.mean(x)) if x.size > 0 else np.nan


def safe_std(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size > 1:
        return float(np.std(x, ddof=1))
    if x.size == 1:
        return 0.0
    return np.nan


def safe_var(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.var(x)) if x.size > 0 else np.nan


def safe_min(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.min(x)) if x.size > 0 else np.nan


def safe_max(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.max(x)) if x.size > 0 else np.nan


def safe_range(x):
    mn = safe_min(x)
    mx = safe_max(x)
    if np.isnan(mn) or np.isnan(mx):
        return np.nan
    return float(mx - mn)


def coefficient_of_variation(x):
    m = safe_mean(x)
    s = safe_std(x)
    if pd.isna(m) or m == 0:
        return np.nan
    return float(s / abs(m))


def linear_slope(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return np.nan
    x = np.arange(len(values))
    return float(np.polyfit(x, values, 1)[0])


def chroma_entropy(chroma):
    chroma_sum = np.sum(np.maximum(chroma, 0), axis=1)
    total = np.sum(chroma_sum)
    if total <= 0:
        return np.nan
    p = chroma_sum / total
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))


def parse_segment_filename(wav_path, root):
    rel_path = wav_path.relative_to(root)
    folder = str(rel_path.parent).replace("\\", "/") if str(rel_path.parent) != "." else ""
    filename = wav_path.name
    stem = wav_path.stem

    if "__seg" in stem:
        parent_track_id = stem.split("__seg")[0]
        seg_part = stem.split("__seg")[-1]
        digits = re.sub(r"\D", "", seg_part)
        segment_index = int(digits) if digits else np.nan
    else:
        parent_track_id = stem
        segment_index = np.nan

    return {
        "segment_filename": filename,
        "segment_stem": stem,
        "relative_path_segment": str(rel_path).replace("\\", "/"),
        "folder": folder,
        "parent_track_id_inferred": parent_track_id,
        "segment_index_within_track_inferred": segment_index
    }


def extract_features_from_wav_fast(wav_path, root):
    info = parse_segment_filename(wav_path, root)

    y, sr = librosa.load(str(wav_path), sr=SR, mono=True)
    duration_sec = librosa.get_duration(y=y, sr=sr)

    if len(y) < int(sr * 0.5):
        raise RuntimeError("Audio segment shorter than 0.5 seconds after loading.")

    rms = librosa.feature.rms(y=y)[0]

    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85)[0]
    spectral_flatness = librosa.feature.spectral_flatness(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y=y)[0]

    onset_env = librosa.onset.onset_strength(y=y, sr=sr)

    try:
        tempo_arr = librosa.feature.tempo(onset_envelope=onset_env, sr=sr)
        tempo_bpm = float(tempo_arr[0]) if len(tempo_arr) > 0 else np.nan
        tempo_valid_flag = 1
    except Exception:
        tempo_bpm = np.nan
        tempo_valid_flag = 0

    try:
        onset_frames = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr)
        onset_count = int(len(onset_frames))
        onset_density = float(onset_count / duration_sec) if duration_sec > 0 else np.nan
    except Exception:
        onset_count = np.nan
        onset_density = np.nan

    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)

    one_sec_features = {
        "rms_1s": [],
        "centroid_1s": [],
        "onset_strength_1s": [],
    }

    samples_per_sec = sr
    n_seconds = int(np.floor(duration_sec))

    for sec in range(n_seconds):
        start = sec * samples_per_sec
        end = min((sec + 1) * samples_per_sec, len(y))
        y_sec = y[start:end]

        if len(y_sec) < int(sr * 0.2):
            continue

        rms_sec = librosa.feature.rms(y=y_sec)[0]
        cent_sec = librosa.feature.spectral_centroid(y=y_sec, sr=sr)[0]
        onset_sec = librosa.onset.onset_strength(y=y_sec, sr=sr)

        one_sec_features["rms_1s"].append(safe_mean(rms_sec))
        one_sec_features["centroid_1s"].append(safe_mean(cent_sec))
        one_sec_features["onset_strength_1s"].append(safe_mean(onset_sec))

    row = {
        **info,
        "duration_sec": float(duration_sec),
        "sampling_rate": sr,

        "rms_mean": safe_mean(rms),
        "rms_std": safe_std(rms),
        "rms_var": safe_var(rms),
        "rms_min": safe_min(rms),
        "rms_max": safe_max(rms),
        "rms_range": safe_range(rms),
        "rms_cv": coefficient_of_variation(rms),

        "spectral_centroid_mean": safe_mean(spectral_centroid),
        "spectral_centroid_std": safe_std(spectral_centroid),
        "spectral_centroid_range": safe_range(spectral_centroid),
        "spectral_bandwidth_mean": safe_mean(spectral_bandwidth),
        "spectral_bandwidth_std": safe_std(spectral_bandwidth),
        "spectral_rolloff_mean": safe_mean(spectral_rolloff),
        "spectral_rolloff_std": safe_std(spectral_rolloff),
        "spectral_flatness_mean": safe_mean(spectral_flatness),
        "spectral_flatness_std": safe_std(spectral_flatness),
        "zero_crossing_rate_mean": safe_mean(zcr),
        "zero_crossing_rate_std": safe_std(zcr),

        "tempo_bpm": tempo_bpm,
        "tempo_valid_flag": tempo_valid_flag,
        "onset_count": onset_count,
        "onset_density": onset_density,
        "onset_strength_mean": safe_mean(onset_env),
        "onset_strength_std": safe_std(onset_env),
        "onset_strength_range": safe_range(onset_env),

        "chroma_mean": safe_mean(chroma.flatten()),
        "chroma_std": safe_std(chroma.flatten()),
        "chroma_var": safe_var(chroma.flatten()),
        "chroma_entropy": chroma_entropy(chroma),

        "n_1s_frames": len(one_sec_features["rms_1s"]),
        "rms_1s_mean": safe_mean(one_sec_features["rms_1s"]),
        "rms_1s_std": safe_std(one_sec_features["rms_1s"]),
        "rms_1s_slope": linear_slope(one_sec_features["rms_1s"]),
        "rms_1s_cv": coefficient_of_variation(one_sec_features["rms_1s"]),
        "centroid_1s_mean": safe_mean(one_sec_features["centroid_1s"]),
        "centroid_1s_std": safe_std(one_sec_features["centroid_1s"]),
        "centroid_1s_slope": linear_slope(one_sec_features["centroid_1s"]),
        "centroid_1s_cv": coefficient_of_variation(one_sec_features["centroid_1s"]),
        "onset_strength_1s_mean": safe_mean(one_sec_features["onset_strength_1s"]),
        "onset_strength_1s_std": safe_std(one_sec_features["onset_strength_1s"]),
        "onset_strength_1s_slope": linear_slope(one_sec_features["onset_strength_1s"]),
        "onset_strength_1s_cv": coefficient_of_variation(one_sec_features["onset_strength_1s"]),

        "feature_extraction_status": "success",
        "feature_extraction_error": ""
    }

    for i in range(N_MFCC):
        row[f"mfcc_{i+1}_mean"] = safe_mean(mfcc[i])
        row[f"mfcc_{i+1}_std"] = safe_std(mfcc[i])

    return row


def main():
    root = Path(SEGMENTED_AUDIO_ROOT)
    wav_files = get_all_wav_files(SEGMENTED_AUDIO_ROOT)

    print("총 WAV 파일 수:", len(wav_files))
    print("feature v2 fast 추출 시작")

    rows = []
    error_rows = []

    for idx, wav_path in enumerate(wav_files, start=1):
        try:
            row = extract_features_from_wav_fast(wav_path, root)
            rows.append(row)
        except Exception as e:
            info = parse_segment_filename(wav_path, root)
            error_rows.append({
                **info,
                "absolute_path": str(wav_path),
                "feature_extraction_status": "error",
                "feature_extraction_error": str(e)
            })

        if idx % 100 == 0:
            print(f"[진행] {idx:,} / {len(wav_files):,}")

    feature_df = pd.DataFrame(rows)
    error_df = pd.DataFrame(error_rows)

    feature_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    error_df.to_csv(ERROR_CSV, index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        feature_df.to_excel(writer, sheet_name="segment_features_v2_fast", index=False)
        error_df.to_excel(writer, sheet_name="errors", index=False)

        summary = pd.DataFrame({
            "item": [
                "total_wav_files",
                "success_rows",
                "error_rows",
                "sampling_rate",
                "n_mfcc",
                "f0_extraction"
            ],
            "value": [
                len(wav_files),
                len(feature_df),
                len(error_df),
                SR,
                N_MFCC,
                "not_included_fast_version"
            ]
        })
        summary.to_excel(writer, sheet_name="summary", index=False)

    print("완료")
    print("성공:", len(feature_df))
    print("오류:", len(error_df))
    print("CSV 저장:", OUTPUT_CSV)
    print("Excel 저장:", OUTPUT_XLSX)
    print("Error 저장:", ERROR_CSV)


if __name__ == "__main__":
    main()

총 WAV 파일 수: 8928
feature v2 fast 추출 시작
[진행] 100 / 8,928
[진행] 200 / 8,928
[진행] 300 / 8,928
[진행] 400 / 8,928
[진행] 500 / 8,928
[진행] 600 / 8,928
[진행] 700 / 8,928
[진행] 800 / 8,928
[진행] 900 / 8,928
[진행] 1,000 / 8,928
[진행] 1,100 / 8,928
[진행] 1,200 / 8,928
[진행] 1,300 / 8,928
[진행] 1,400 / 8,928
[진행] 1,500 / 8,928
[진행] 1,600 / 8,928
[진행] 1,700 / 8,928
[진행] 1,800 / 8,928
[진행] 1,900 / 8,928
[진행] 2,000 / 8,928
[진행] 2,100 / 8,928
[진행] 2,200 / 8,928
[진행] 2,300 / 8,928
[진행] 2,400 / 8,928
[진행] 2,500 / 8,928
[진행] 2,600 / 8,928
[진행] 2,700 / 8,928
[진행] 2,800 / 8,928
[진행] 2,900 / 8,928
[진행] 3,000 / 8,928
[진행] 3,100 / 8,928
[진행] 3,200 / 8,928
[진행] 3,300 / 8,928
[진행] 3,400 / 8,928
[진행] 3,500 / 8,928
[진행] 3,600 / 8,928
[진행] 3,700 / 8,928
[진행] 3,800 / 8,928
[진행] 3,900 / 8,928
[진행] 4,000 / 8,928
[진행] 4,100 / 8,928
[진행] 4,200 / 8,928
[진행] 4,300 / 8,928
[진행] 4,400 / 8,928
[진행] 4,500 / 8,928
[진행] 4,600 / 8,928
[진행] 4,700 / 8,928
[진행] 4,800 / 8,928
[진행] 4,900 / 8,928
[진행] 5,000 / 8,928
[진행] 5,100 / 8,928
[진행] 5,200

In [15]:
import os
import pandas as pd
import numpy as np

# ============================================================
# 0. 경로 설정
# ============================================================

FEATURE_CSV = r"D:\gugak_scientific_data_package_v1\data\segment_acoustic_features_v2_fast.csv"

METADATA_XLSX = r"D:\gugak_metadata_labeling\segmented_wav_metadata_refined_v5_remaining_fixed.xlsx"

OUTPUT_DIR = r"D:\gugak_scientific_data_package_v1"
DATA_DIR = os.path.join(OUTPUT_DIR, "data")
METADATA_DIR = os.path.join(OUTPUT_DIR, "metadata")
VALIDATION_DIR = os.path.join(OUTPUT_DIR, "validation")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(METADATA_DIR, exist_ok=True)
os.makedirs(VALIDATION_DIR, exist_ok=True)


# ============================================================
# 1. 파일 불러오기
# ============================================================

print("1단계: feature v2 CSV 불러오기")

feature_df = pd.read_csv(FEATURE_CSV, encoding="utf-8-sig")

print("feature_df 행 수:", len(feature_df))
print("feature_df 컬럼 수:", len(feature_df.columns))


print("\n2단계: refined metadata 불러오기")

metadata_df = pd.read_excel(METADATA_XLSX, sheet_name="refined_metadata")

print("metadata_df 행 수:", len(metadata_df))
print("metadata_df 컬럼 수:", len(metadata_df.columns))


# ============================================================
# 2. 병합 키 정리
# ============================================================

print("\n3단계: 병합 키 생성")

feature_df["merge_key_filename"] = feature_df["segment_filename"].astype(str).str.strip()
metadata_df["merge_key_filename"] = metadata_df["segment_filename"].astype(str).str.strip()

print("feature 고유 filename 수:", feature_df["merge_key_filename"].nunique())
print("metadata 고유 filename 수:", metadata_df["merge_key_filename"].nunique())

feature_dup = feature_df["merge_key_filename"].duplicated().sum()
metadata_dup = metadata_df["merge_key_filename"].duplicated().sum()

print("feature 중복 key 수:", feature_dup)
print("metadata 중복 key 수:", metadata_dup)


# ============================================================
# 3. metadata에서 필요한 컬럼만 선택
# ============================================================

metadata_keep_cols = [
    "merge_key_filename",
    "segment_file_id",
    "segment_filename",
    "segment_stem",
    "relative_path_segment",
    "folder",
    "parent_track_id_inferred",
    "segment_index_within_track_inferred",
    "duration_sec",
    "sampling_rate_detected",
    "audio_read_status",
    "audio_error",

    "instrument_category_refined",
    "primary_instrument_refined",
    "genre_level1_refined",
    "genre_level2_refined",
    "vocal_instrumental_type_refined",
    "ensemble_flag",
    "jangdan_inferred",
    "related_repertoire_inferred",

    "folder_refinement_rules",
    "metadata_refinement_status",
    "manual_review_needed_refined",
    "manual_review_reason_refined",
]

metadata_keep_cols = [c for c in metadata_keep_cols if c in metadata_df.columns]
metadata_small = metadata_df[metadata_keep_cols].copy()

print("\nmetadata 병합용 컬럼 수:", len(metadata_small.columns))


# ============================================================
# 4. 병합
# ============================================================

print("\n4단계: feature + metadata 병합")

merged_df = feature_df.merge(
    metadata_small,
    on="merge_key_filename",
    how="left",
    suffixes=("_feature", "_metadata"),
    validate="one_to_one"
)

print("병합 후 행 수:", len(merged_df))
print("병합 후 컬럼 수:", len(merged_df.columns))


# ============================================================
# 5. 병합 검증
# ============================================================

print("\n5단계: 병합 검증")

# metadata가 정상적으로 붙었는지 확인
merged_df["metadata_matched"] = merged_df["instrument_category_refined"].notna().astype(int)

n_total = len(merged_df)
n_matched = int(merged_df["metadata_matched"].sum())
n_unmatched = n_total - n_matched

print("전체 feature 행:", n_total)
print("metadata 매칭 성공:", n_matched)
print("metadata 매칭 실패:", n_unmatched)
print("매칭률(%):", round(n_matched / n_total * 100, 2))

# unmatched 저장
unmatched_df = merged_df[merged_df["metadata_matched"] == 0].copy()

if len(unmatched_df) > 0:
    unmatched_path = os.path.join(VALIDATION_DIR, "unmatched_feature_metadata_rows_v2.xlsx")
    unmatched_df.to_excel(unmatched_path, index=False)
    print("매칭 실패 파일 저장:", unmatched_path)

# metadata에는 있는데 feature에는 없는 파일
feature_keys = set(feature_df["merge_key_filename"])
metadata_keys = set(metadata_df["merge_key_filename"])

feature_only = sorted(list(feature_keys - metadata_keys))
metadata_only = sorted(list(metadata_keys - feature_keys))

feature_only_df = feature_df[feature_df["merge_key_filename"].isin(feature_only)].copy()
metadata_only_df = metadata_df[metadata_df["merge_key_filename"].isin(metadata_only)].copy()

feature_only_path = os.path.join(VALIDATION_DIR, "feature_only_rows_v2.xlsx")
metadata_only_path = os.path.join(VALIDATION_DIR, "metadata_only_rows_v2.xlsx")

feature_only_df.to_excel(feature_only_path, index=False)
metadata_only_df.to_excel(metadata_only_path, index=False)

print("feature에만 있는 행 수:", len(feature_only_df))
print("metadata에만 있는 행 수:", len(metadata_only_df))


# ============================================================
# 6. 컬럼명 정리
# ============================================================

print("\n6단계: 컬럼명 정리")

# duration_sec가 feature와 metadata 양쪽에 있어서 suffix가 붙었을 수 있음
rename_map = {
    "duration_sec_feature": "segment_duration_sec_feature",
    "duration_sec_metadata": "segment_duration_sec_metadata",
    "relative_path_segment_feature": "relative_path_segment_feature",
    "relative_path_segment_metadata": "relative_path_segment_metadata",
    "folder_feature": "folder_feature",
    "folder_metadata": "folder_metadata",
    "parent_track_id_inferred_feature": "parent_track_id_feature",
    "parent_track_id_inferred_metadata": "parent_track_id_metadata",
    "segment_index_within_track_inferred_feature": "segment_index_feature",
    "segment_index_within_track_inferred_metadata": "segment_index_metadata",
}

existing_rename_map = {k: v for k, v in rename_map.items() if k in merged_df.columns}
merged_df = merged_df.rename(columns=existing_rename_map)

print("변경된 컬럼:", existing_rename_map)


# ============================================================
# 7. 최종 segment table 컬럼 순서 정리
# ============================================================

print("\n7단계: 최종 컬럼 순서 정리")

id_cols = [
    "segment_file_id",
    "segment_filename_feature",
    "segment_filename_metadata",
    "segment_stem_feature",
    "segment_stem_metadata",
    "merge_key_filename",
    "relative_path_segment_feature",
    "relative_path_segment_metadata",
    "folder_feature",
    "folder_metadata",
    "parent_track_id_feature",
    "parent_track_id_metadata",
    "segment_index_feature",
    "segment_index_metadata",
    "segment_duration_sec_feature",
    "segment_duration_sec_metadata",
    "sampling_rate",
    "sampling_rate_detected",
    "metadata_matched",
]

metadata_cols = [
    "instrument_category_refined",
    "primary_instrument_refined",
    "genre_level1_refined",
    "genre_level2_refined",
    "vocal_instrumental_type_refined",
    "ensemble_flag",
    "jangdan_inferred",
    "related_repertoire_inferred",
    "folder_refinement_rules",
    "metadata_refinement_status",
    "manual_review_needed_refined",
    "manual_review_reason_refined",
]

feature_cols = [
    "rms_mean",
    "rms_std",
    "rms_var",
    "rms_min",
    "rms_max",
    "rms_range",
    "rms_cv",

    "spectral_centroid_mean",
    "spectral_centroid_std",
    "spectral_centroid_range",
    "spectral_bandwidth_mean",
    "spectral_bandwidth_std",
    "spectral_rolloff_mean",
    "spectral_rolloff_std",
    "spectral_flatness_mean",
    "spectral_flatness_std",
    "zero_crossing_rate_mean",
    "zero_crossing_rate_std",

    "tempo_bpm",
    "tempo_valid_flag",
    "onset_count",
    "onset_density",
    "onset_strength_mean",
    "onset_strength_std",
    "onset_strength_range",

    "chroma_mean",
    "chroma_std",
    "chroma_var",
    "chroma_entropy",

    "n_1s_frames",
    "rms_1s_mean",
    "rms_1s_std",
    "rms_1s_slope",
    "rms_1s_cv",
    "centroid_1s_mean",
    "centroid_1s_std",
    "centroid_1s_slope",
    "centroid_1s_cv",
    "onset_strength_1s_mean",
    "onset_strength_1s_std",
    "onset_strength_1s_slope",
    "onset_strength_1s_cv",
]

mfcc_cols = []
for i in range(1, 14):
    mfcc_cols.append(f"mfcc_{i}_mean")
    mfcc_cols.append(f"mfcc_{i}_std")

status_cols = [
    "feature_extraction_status",
    "feature_extraction_error",
    "audio_read_status",
    "audio_error",
]

ordered_cols = id_cols + metadata_cols + feature_cols + mfcc_cols + status_cols
ordered_cols = [c for c in ordered_cols if c in merged_df.columns]

remaining_cols = [c for c in merged_df.columns if c not in ordered_cols]
merged_df = merged_df[ordered_cols + remaining_cols]


# ============================================================
# 8. track-level summary 생성
# ============================================================

print("\n8단계: track-level summary 생성")

track_col = "parent_track_id_feature" if "parent_track_id_feature" in merged_df.columns else "parent_track_id_inferred"

if track_col in merged_df.columns:
    track_summary = (
        merged_df
        .groupby(track_col)
        .agg(
            n_segments=("merge_key_filename", "count"),
            total_segment_duration_sec=("segment_duration_sec_feature", "sum"),
            mean_segment_duration_sec=("segment_duration_sec_feature", "mean"),
            instrument_category=("instrument_category_refined", lambda x: ";".join(sorted(set(x.dropna().astype(str))))),
            primary_instrument=("primary_instrument_refined", lambda x: ";".join(sorted(set(x.dropna().astype(str))))),
            genre_level1=("genre_level1_refined", lambda x: ";".join(sorted(set(x.dropna().astype(str))))),
            genre_level2=("genre_level2_refined", lambda x: ";".join(sorted(set(x.dropna().astype(str))))),
            vocal_instrumental_type=("vocal_instrumental_type_refined", lambda x: ";".join(sorted(set(x.dropna().astype(str))))),
            rms_mean=("rms_mean", "mean"),
            spectral_centroid_mean=("spectral_centroid_mean", "mean"),
            onset_density_mean=("onset_density", "mean"),
            tempo_bpm_mean=("tempo_bpm", "mean"),
        )
        .reset_index()
        .rename(columns={track_col: "parent_track_id"})
    )
else:
    track_summary = pd.DataFrame()


# ============================================================
# 9. validation summary 생성
# ============================================================

print("\n9단계: validation summary 생성")

validation_summary = pd.DataFrame({
    "item": [
        "feature_rows",
        "metadata_rows",
        "merged_rows",
        "metadata_matched_rows",
        "metadata_unmatched_rows",
        "metadata_match_rate_percent",
        "feature_unique_filenames",
        "metadata_unique_filenames",
        "feature_duplicate_keys",
        "metadata_duplicate_keys",
        "feature_only_rows",
        "metadata_only_rows",
        "feature_extraction_success_rows",
        "feature_extraction_error_rows",
        "final_manual_review_needed_rows",
    ],
    "value": [
        len(feature_df),
        len(metadata_df),
        len(merged_df),
        n_matched,
        n_unmatched,
        round(n_matched / n_total * 100, 2),
        feature_df["merge_key_filename"].nunique(),
        metadata_df["merge_key_filename"].nunique(),
        int(feature_dup),
        int(metadata_dup),
        len(feature_only_df),
        len(metadata_only_df),
        int((merged_df["feature_extraction_status"] == "success").sum()) if "feature_extraction_status" in merged_df.columns else np.nan,
        int((merged_df["feature_extraction_status"] == "error").sum()) if "feature_extraction_status" in merged_df.columns else np.nan,
        int(merged_df["manual_review_needed_refined"].fillna(1).sum()) if "manual_review_needed_refined" in merged_df.columns else np.nan,
    ]
})


# ============================================================
# 10. data dictionary 생성
# ============================================================

print("\n10단계: data dictionary 생성")

def describe_column(col):
    desc = {
        "segment_file_id": "Unique identifier for each segment file.",
        "merge_key_filename": "Filename-based key used to merge acoustic features and metadata.",
        "segment_filename_feature": "Segment WAV filename from the feature extraction table.",
        "segment_filename_metadata": "Segment WAV filename from the metadata table.",
        "relative_path_segment_feature": "Relative path to the segment WAV file from the feature table.",
        "relative_path_segment_metadata": "Relative path to the segment WAV file from the metadata table.",
        "folder_feature": "Folder path inferred during feature extraction.",
        "folder_metadata": "Folder path recorded in the metadata table.",
        "parent_track_id_feature": "Parent track identifier inferred during feature extraction.",
        "parent_track_id_metadata": "Parent track identifier inferred during metadata generation.",
        "segment_index_feature": "Segment order within parent track inferred during feature extraction.",
        "segment_index_metadata": "Segment order within parent track inferred during metadata generation.",
        "segment_duration_sec_feature": "Segment duration estimated during feature extraction.",
        "segment_duration_sec_metadata": "Segment duration estimated during metadata generation.",
        "sampling_rate": "Sampling rate used for feature extraction.",
        "sampling_rate_detected": "Sampling rate detected during metadata generation.",
        "metadata_matched": "Indicator for successful metadata matching.",
        "instrument_category_refined": "Rule-based inferred instrument category.",
        "primary_instrument_refined": "Rule-based inferred primary instrument.",
        "genre_level1_refined": "Rule-based inferred broad genre category.",
        "genre_level2_refined": "Rule-based inferred detailed genre or repertoire category.",
        "vocal_instrumental_type_refined": "Rule-based inferred vocal or instrumental type.",
        "ensemble_flag": "Rule-based inferred ensemble indicator.",
        "jangdan_inferred": "Rhythmic-cycle or jangdan term inferred from folder or filename.",
        "related_repertoire_inferred": "Related repertoire term inferred from folder or filename.",
        "folder_refinement_rules": "Provenance field indicating which rule assigned or refined metadata.",
        "metadata_refinement_status": "Status of rule-based metadata refinement.",
        "manual_review_needed_refined": "Whether unresolved metadata remained after final rule-based refinement.",
        "manual_review_reason_refined": "Reason for unresolved metadata after final rule-based refinement.",
        "rms_mean": "Mean root-mean-square energy.",
        "rms_std": "Standard deviation of root-mean-square energy.",
        "rms_var": "Variance of root-mean-square energy.",
        "rms_range": "Range of root-mean-square energy.",
        "rms_cv": "Coefficient of variation of root-mean-square energy.",
        "spectral_centroid_mean": "Mean spectral centroid.",
        "spectral_centroid_std": "Standard deviation of spectral centroid.",
        "spectral_centroid_range": "Range of spectral centroid.",
        "spectral_bandwidth_mean": "Mean spectral bandwidth.",
        "spectral_bandwidth_std": "Standard deviation of spectral bandwidth.",
        "spectral_rolloff_mean": "Mean spectral rolloff.",
        "spectral_rolloff_std": "Standard deviation of spectral rolloff.",
        "spectral_flatness_mean": "Mean spectral flatness.",
        "spectral_flatness_std": "Standard deviation of spectral flatness.",
        "zero_crossing_rate_mean": "Mean zero-crossing rate.",
        "zero_crossing_rate_std": "Standard deviation of zero-crossing rate.",
        "tempo_bpm": "Estimated tempo in beats per minute.",
        "tempo_valid_flag": "Indicator for successful tempo estimation.",
        "onset_count": "Number of detected onset events.",
        "onset_density": "Detected onsets per second.",
        "onset_strength_mean": "Mean onset strength.",
        "onset_strength_std": "Standard deviation of onset strength.",
        "onset_strength_range": "Range of onset strength.",
        "chroma_mean": "Mean chroma feature value.",
        "chroma_std": "Standard deviation of chroma feature value.",
        "chroma_var": "Variance of chroma feature value.",
        "chroma_entropy": "Entropy of pitch-class distribution derived from chroma features.",
        "n_1s_frames": "Number of 1-second subframes used for within-segment dynamics.",
        "rms_1s_slope": "Linear slope of 1-second RMS values within the segment.",
        "centroid_1s_slope": "Linear slope of 1-second spectral centroid values within the segment.",
        "onset_strength_1s_slope": "Linear slope of 1-second onset strength values within the segment.",
        "feature_extraction_status": "Status of acoustic feature extraction.",
        "feature_extraction_error": "Error message from feature extraction, if any.",
    }

    if col.startswith("mfcc_"):
        return "Mel-frequency cepstral coefficient summary feature."

    return desc.get(col, "Column generated during the Gugak acoustic dataset construction pipeline.")


data_dictionary = pd.DataFrame({
    "column_name": merged_df.columns,
    "data_type": [str(merged_df[c].dtype) for c in merged_df.columns],
    "missing_count": [int(merged_df[c].isna().sum()) for c in merged_df.columns],
    "missing_rate_percent": [round(merged_df[c].isna().mean() * 100, 4) for c in merged_df.columns],
    "description": [describe_column(c) for c in merged_df.columns],
})


# ============================================================
# 11. 저장
# ============================================================

print("\n11단계: 파일 저장")

segment_full_csv = os.path.join(DATA_DIR, "segment_acoustic_features_with_metadata_v2.csv")
segment_full_xlsx = os.path.join(DATA_DIR, "segment_acoustic_features_with_metadata_v2.xlsx")
track_summary_csv = os.path.join(METADATA_DIR, "track_metadata_summary_v2.csv")
validation_csv = os.path.join(VALIDATION_DIR, "merge_validation_summary_v2.csv")
data_dictionary_csv = os.path.join(METADATA_DIR, "data_dictionary_segment_acoustic_features_with_metadata_v2.csv")

merged_df.to_csv(segment_full_csv, index=False, encoding="utf-8-sig")
track_summary.to_csv(track_summary_csv, index=False, encoding="utf-8-sig")
validation_summary.to_csv(validation_csv, index=False, encoding="utf-8-sig")
data_dictionary.to_csv(data_dictionary_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(segment_full_xlsx, engine="openpyxl") as writer:
    merged_df.to_excel(writer, sheet_name="segment_full_v2", index=False)
    track_summary.to_excel(writer, sheet_name="track_summary_v2", index=False)
    validation_summary.to_excel(writer, sheet_name="validation_summary", index=False)
    data_dictionary.to_excel(writer, sheet_name="data_dictionary", index=False)


# ============================================================
# 12. 완료
# ============================================================

print("\n===================================================")
print("완료: feature v2 + metadata v5 병합")
print("===================================================")
print("전체 병합 CSV:", segment_full_csv)
print("전체 병합 Excel:", segment_full_xlsx)
print("track summary:", track_summary_csv)
print("validation summary:", validation_csv)
print("data dictionary:", data_dictionary_csv)
print()
print(validation_summary)


1단계: feature v2 CSV 불러오기
feature_df 행 수: 8928
feature_df 컬럼 수: 78

2단계: refined metadata 불러오기
metadata_df 행 수: 8928
metadata_df 컬럼 수: 37

3단계: 병합 키 생성
feature 고유 filename 수: 8928
metadata 고유 filename 수: 8928
feature 중복 key 수: 0
metadata 중복 key 수: 0

metadata 병합용 컬럼 수: 24

4단계: feature + metadata 병합
병합 후 행 수: 8928
병합 후 컬럼 수: 102

5단계: 병합 검증
전체 feature 행: 8928
metadata 매칭 성공: 8928
metadata 매칭 실패: 0
매칭률(%): 100.0
feature에만 있는 행 수: 0
metadata에만 있는 행 수: 0

6단계: 컬럼명 정리
변경된 컬럼: {'duration_sec_feature': 'segment_duration_sec_feature', 'duration_sec_metadata': 'segment_duration_sec_metadata', 'relative_path_segment_feature': 'relative_path_segment_feature', 'relative_path_segment_metadata': 'relative_path_segment_metadata', 'folder_feature': 'folder_feature', 'folder_metadata': 'folder_metadata', 'parent_track_id_inferred_feature': 'parent_track_id_feature', 'parent_track_id_inferred_metadata': 'parent_track_id_metadata', 'segment_index_within_track_inferred_feature': 'segment_index_feature', '

In [16]:
# ============================================================
# Gugak Acoustic Yin–Yang Orientation Profile v1
# ------------------------------------------------------------
# 목적:
#   1) segment-level acoustic feature + metadata table 불러오기
#   2) 음악 요소별 acoustic orientation 계산
#   3) overall acoustic orientation index 계산
#   4) orientation profile group 생성
#   5) track-level temporal trajectory 생성
#   6) Scientific Data용 CSV / Excel / validation / figure 저장
#
# 중요:
#   이 값은 "음/양 정답 라벨"이 아닙니다.
#   재현 가능한 음향 feature를 기반으로 한
#   "Acoustic Yin–Yang Orientation Profile"입니다.
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# 0. 경로 설정
# ============================================================

INPUT_CSV = r"D:\gugak_scientific_data_package_v1\data\segment_acoustic_features_with_metadata_v2.csv"

OUTPUT_DIR = r"D:\gugak_scientific_data_package_v1"
DATA_DIR = os.path.join(OUTPUT_DIR, "data")
ORIENTATION_DIR = os.path.join(OUTPUT_DIR, "orientation_profiles")
VALIDATION_DIR = os.path.join(OUTPUT_DIR, "validation")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
FIGURE_DIR = os.path.join(OUTPUT_DIR, "figures")
METADATA_DIR = os.path.join(OUTPUT_DIR, "metadata")

for d in [DATA_DIR, ORIENTATION_DIR, VALIDATION_DIR, TABLE_DIR, FIGURE_DIR, METADATA_DIR]:
    os.makedirs(d, exist_ok=True)

VERSION = "YYOP_v1.0_acoustic_orientation_profile"


# ============================================================
# 1. 데이터 불러오기
# ============================================================

print("1단계: 병합된 segment dataset 불러오기")

df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")

print("행 수:", len(df))
print("컬럼 수:", len(df.columns))

if len(df) != 8928:
    print("주의: 행 수가 8,928개가 아닙니다. 현재 행 수:", len(df))
else:
    print("확인: 8,928개 segment가 정상적으로 불러와졌습니다.")


# ============================================================
# 2. 유틸 함수
# ============================================================

def to_numeric_series(s):
    return pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)


def winsorize_series(s, lower_q=0.01, upper_q=0.99):
    s = to_numeric_series(s)
    lower = s.quantile(lower_q)
    upper = s.quantile(upper_q)
    return s.clip(lower=lower, upper=upper)


def robust_zscore_dataframe(df_in, cols, lower_q=0.01, upper_q=0.99):
    """
    1) 숫자 변환
    2) inf 제거
    3) median 대체
    4) 1~99 percentile winsorization
    5) z-score 표준화

    반환:
      z_df: 표준화된 feature table
      stats_df: 각 feature의 처리 정보
    """
    z_data = {}
    stats_rows = []

    for col in cols:
        s_raw = to_numeric_series(df_in[col])
        missing_before = int(s_raw.isna().sum())

        median_val = s_raw.median()
        s_filled = s_raw.fillna(median_val)

        lower = s_filled.quantile(lower_q)
        upper = s_filled.quantile(upper_q)
        s_win = s_filled.clip(lower=lower, upper=upper)

        mean_val = s_win.mean()
        std_val = s_win.std(ddof=1)

        if pd.isna(std_val) or std_val == 0:
            z = pd.Series(np.zeros(len(s_win)), index=s_win.index)
            std_used = 0.0
        else:
            z = (s_win - mean_val) / std_val
            std_used = std_val

        z_data[col] = z

        stats_rows.append({
            "feature": col,
            "missing_before_imputation": missing_before,
            "median_imputed_value": median_val,
            "winsor_lower_q": lower_q,
            "winsor_upper_q": upper_q,
            "winsor_lower_value": lower,
            "winsor_upper_value": upper,
            "mean_after_winsorization": mean_val,
            "std_after_winsorization": std_used
        })

    z_df = pd.DataFrame(z_data, index=df_in.index)
    stats_df = pd.DataFrame(stats_rows)

    return z_df, stats_df


def scale_to_minus1_plus1(s, lower_q=0.01, upper_q=0.99):
    """
    raw orientation 값을 -1 ~ +1 범위로 변환.
    극단값 영향을 줄이기 위해 1~99 percentile 기준으로 scaling.
    """
    s = to_numeric_series(s)
    p_low = s.quantile(lower_q)
    p_high = s.quantile(upper_q)

    if pd.isna(p_low) or pd.isna(p_high) or p_high == p_low:
        return pd.Series(np.zeros(len(s)), index=s.index), p_low, p_high

    scaled = 2 * ((s - p_low) / (p_high - p_low)) - 1
    scaled = scaled.clip(-1, 1)

    return scaled, p_low, p_high


def safe_unique_join(x):
    vals = sorted(set(x.dropna().astype(str)))
    vals = [v for v in vals if v not in ["nan", "None", ""]]
    return ";".join(vals)


def linear_slope_normalized(y_values):
    """
    track 내 segment 순서를 0~1로 정규화한 뒤 orientation 기울기 계산.
    slope > 0: 시간 경과에 따라 Yang 방향 증가
    slope < 0: 시간 경과에 따라 Yin 방향 증가
    """
    y = np.asarray(y_values, dtype=float)
    mask = np.isfinite(y)
    y = y[mask]

    if len(y) < 2:
        return np.nan

    x = np.linspace(0, 1, len(y))
    slope = np.polyfit(x, y, 1)[0]
    return float(slope)


# ============================================================
# 3. Orientation mapping rule 정의
# ============================================================
# 원칙:
#   값이 높을수록 활동성, 강도, 밝기, 변동성, 분절성이 커지는 feature를
#   Yang-oriented 방향으로 둡니다.
#
# 주의:
#   이 mapping은 "음양의 정답"이 아니라,
#   국악/동양음악 이론에서 논의되는 음악적 속성을
#   재현 가능한 음향 feature로 조작화한 규칙입니다.
# ============================================================

domain_feature_map = {
    "rhythmic_activity_orientation": [
        "onset_density",
        "onset_count",
        "onset_strength_mean",
        "onset_strength_std",
        "onset_strength_range"
    ],

    "dynamic_intensity_orientation": [
        "rms_mean",
        "rms_std",
        "rms_var",
        "rms_range",
        "rms_cv"
    ],

    "tempo_activity_orientation": [
        "tempo_bpm"
    ],

    "spectral_brightness_orientation": [
        "spectral_centroid_mean",
        "spectral_centroid_std",
        "spectral_bandwidth_mean",
        "spectral_rolloff_mean",
        "spectral_flatness_mean",
        "zero_crossing_rate_mean"
    ],

    "tonal_variability_orientation": [
        "chroma_std",
        "chroma_var",
        "chroma_entropy"
    ],

    "within_segment_dynamics_orientation": [
        "rms_1s_std",
        "rms_1s_cv",
        "centroid_1s_std",
        "centroid_1s_cv",
        "onset_strength_1s_std",
        "onset_strength_1s_cv"
    ]
}

# 실제 데이터에 존재하는 feature만 사용
available_domain_feature_map = {}

for domain, features in domain_feature_map.items():
    available = [f for f in features if f in df.columns]
    available_domain_feature_map[domain] = available

print("\n2단계: domain별 사용 가능한 feature 확인")
for domain, features in available_domain_feature_map.items():
    print(domain, ":", features)

# feature가 하나도 없는 domain이 있으면 경고
empty_domains = [d for d, f in available_domain_feature_map.items() if len(f) == 0]

if empty_domains:
    print("\n주의: 다음 domain에는 사용 가능한 feature가 없습니다.")
    print(empty_domains)


# ============================================================
# 4. Mapping rule table 저장
# ============================================================

mapping_rows = []

feature_interpretation = {
    "onset_density": "Higher onset density indicates more frequent rhythmic articulation.",
    "onset_count": "Higher onset count indicates more segmented rhythmic events.",
    "onset_strength_mean": "Higher onset strength indicates stronger rhythmic attacks.",
    "onset_strength_std": "Higher onset strength variability indicates greater rhythmic fluctuation.",
    "onset_strength_range": "Higher onset strength range indicates greater rhythmic contrast.",

    "rms_mean": "Higher RMS indicates greater acoustic energy.",
    "rms_std": "Higher RMS variability indicates greater dynamic fluctuation.",
    "rms_var": "Higher RMS variance indicates greater dynamic fluctuation.",
    "rms_range": "Higher RMS range indicates greater dynamic contrast.",
    "rms_cv": "Higher RMS coefficient of variation indicates greater relative dynamic fluctuation.",

    "tempo_bpm": "Higher estimated tempo indicates faster temporal motion.",

    "spectral_centroid_mean": "Higher spectral centroid indicates brighter spectral quality.",
    "spectral_centroid_std": "Higher spectral centroid variability indicates greater timbral fluctuation.",
    "spectral_bandwidth_mean": "Higher spectral bandwidth indicates broader spectral spread.",
    "spectral_rolloff_mean": "Higher spectral rolloff indicates greater high-frequency emphasis.",
    "spectral_flatness_mean": "Higher spectral flatness indicates noisier or less tonal spectral quality.",
    "zero_crossing_rate_mean": "Higher zero-crossing rate indicates greater high-frequency or noisy components.",

    "chroma_std": "Higher chroma variability indicates greater pitch-class fluctuation.",
    "chroma_var": "Higher chroma variance indicates greater pitch-class fluctuation.",
    "chroma_entropy": "Higher chroma entropy indicates more dispersed pitch-class distribution.",

    "rms_1s_std": "Higher 1-second RMS variability indicates stronger within-segment dynamic change.",
    "rms_1s_cv": "Higher 1-second RMS coefficient of variation indicates stronger relative dynamic change.",
    "centroid_1s_std": "Higher 1-second centroid variability indicates stronger within-segment timbral change.",
    "centroid_1s_cv": "Higher 1-second centroid coefficient of variation indicates stronger relative timbral change.",
    "onset_strength_1s_std": "Higher 1-second onset-strength variability indicates stronger within-segment rhythmic change.",
    "onset_strength_1s_cv": "Higher 1-second onset-strength coefficient of variation indicates stronger relative rhythmic change."
}

for domain, features in available_domain_feature_map.items():
    for f in features:
        mapping_rows.append({
            "orientation_domain": domain,
            "feature": f,
            "direction": "higher_value_more_yang_oriented",
            "interpretation": feature_interpretation.get(
                f,
                "Higher value was operationally mapped to a more Yang-oriented acoustic tendency."
            ),
            "orientation_algorithm_version": VERSION
        })

mapping_rules_df = pd.DataFrame(mapping_rows)

mapping_rules_path = os.path.join(ORIENTATION_DIR, "orientation_mapping_rules_v1.csv")
mapping_rules_df.to_csv(mapping_rules_path, index=False, encoding="utf-8-sig")

print("\nMapping rule 저장:", mapping_rules_path)


# ============================================================
# 5. Domain-level orientation 계산
# ============================================================

print("\n3단계: domain-level orientation 계산 시작")

orientation_df = df.copy()
all_feature_stats = []
domain_scaling_rows = []

domain_orientation_cols = []

for domain, features in available_domain_feature_map.items():
    if len(features) == 0:
        continue

    z_df, stats_df = robust_zscore_dataframe(orientation_df, features)
    stats_df["orientation_domain"] = domain
    all_feature_stats.append(stats_df)

    # domain raw = 해당 domain feature들의 z-score 평균
    raw_col = domain.replace("_orientation", "_raw")
    orientation_df[raw_col] = z_df.mean(axis=1)

    # -1 ~ +1 scaling
    orientation_col = domain
    scaled, p_low, p_high = scale_to_minus1_plus1(orientation_df[raw_col])

    orientation_df[orientation_col] = scaled
    domain_orientation_cols.append(orientation_col)

    domain_scaling_rows.append({
        "orientation_domain": domain,
        "raw_column": raw_col,
        "scaled_column": orientation_col,
        "features_used": ";".join(features),
        "n_features_used": len(features),
        "scaling_lower_q": 0.01,
        "scaling_upper_q": 0.99,
        "raw_p01": p_low,
        "raw_p99": p_high,
        "orientation_algorithm_version": VERSION
    })

feature_standardization_stats = pd.concat(all_feature_stats, ignore_index=True)
domain_scaling_df = pd.DataFrame(domain_scaling_rows)

print("생성된 domain orientation columns:")
print(domain_orientation_cols)


# ============================================================
# 6. Overall acoustic orientation index 계산
# ============================================================

print("\n4단계: overall acoustic orientation index 계산")

orientation_df["overall_acoustic_orientation_index"] = orientation_df[domain_orientation_cols].mean(axis=1)

orientation_df["domain_orientation_mean"] = orientation_df[domain_orientation_cols].mean(axis=1)
orientation_df["domain_orientation_sd"] = orientation_df[domain_orientation_cols].std(axis=1)
orientation_df["domain_orientation_min"] = orientation_df[domain_orientation_cols].min(axis=1)
orientation_df["domain_orientation_max"] = orientation_df[domain_orientation_cols].max(axis=1)
orientation_df["domain_orientation_range"] = (
    orientation_df["domain_orientation_max"] - orientation_df["domain_orientation_min"]
)

# domain 방향 일관성
positive_cut = 0.15
negative_cut = -0.15

orientation_df["positive_domain_count"] = (orientation_df[domain_orientation_cols] > positive_cut).sum(axis=1)
orientation_df["negative_domain_count"] = (orientation_df[domain_orientation_cols] < negative_cut).sum(axis=1)
orientation_df["neutral_domain_count"] = len(domain_orientation_cols) - (
    orientation_df["positive_domain_count"] + orientation_df["negative_domain_count"]
)

orientation_df["domain_consistency_index"] = (
    (orientation_df["positive_domain_count"] - orientation_df["negative_domain_count"]).abs()
    / len(domain_orientation_cols)
)

orientation_df["orientation_algorithm_version"] = VERSION


# ============================================================
# 7. Segment-level orientation group 생성
# ============================================================

print("\n5단계: segment-level orientation profile group 생성")

def classify_orientation_profile(row):
    overall = row["overall_acoustic_orientation_index"]
    sd = row["domain_orientation_sd"]
    pos = row["positive_domain_count"]
    neg = row["negative_domain_count"]

    # 기준값
    yang_cut = 0.30
    yin_cut = -0.30
    balanced_cut = 0.20
    low_disagreement_cut = 0.35

    # domain 수가 6개라고 가정하면 4개 이상 같은 방향이면 비교적 일관됨
    consistent_domain_count = max(3, int(np.ceil(len(domain_orientation_cols) * 0.60)))

    if overall >= yang_cut and pos >= consistent_domain_count:
        return "Yang-oriented"

    if overall <= yin_cut and neg >= consistent_domain_count:
        return "Yin-oriented"

    if abs(overall) < balanced_cut and sd < low_disagreement_cut:
        return "Balanced"

    return "Mixed-profile"


orientation_df["orientation_profile_group"] = orientation_df.apply(classify_orientation_profile, axis=1)

group_counts = (
    orientation_df["orientation_profile_group"]
    .value_counts()
    .reset_index()
)
group_counts.columns = ["orientation_profile_group", "n"]
group_counts["percent"] = group_counts["n"] / len(orientation_df) * 100

print("\nOrientation profile group 분포:")
print(group_counts)


# ============================================================
# 8. Segment orientation profile table 생성
# ============================================================

segment_id_cols = [
    "segment_file_id",
    "merge_key_filename",
    "segment_filename_feature",
    "segment_filename_metadata",
    "relative_path_segment_feature",
    "relative_path_segment_metadata",
    "folder_feature",
    "folder_metadata",
    "parent_track_id_feature",
    "parent_track_id_metadata",
    "segment_index_feature",
    "segment_index_metadata",
    "segment_duration_sec_feature",
    "segment_duration_sec_metadata",
]

segment_metadata_cols = [
    "instrument_category_refined",
    "primary_instrument_refined",
    "genre_level1_refined",
    "genre_level2_refined",
    "vocal_instrumental_type_refined",
    "ensemble_flag",
    "jangdan_inferred",
    "related_repertoire_inferred",
    "folder_refinement_rules",
    "metadata_refinement_status"
]

segment_orientation_cols = domain_orientation_cols + [
    "overall_acoustic_orientation_index",
    "orientation_profile_group",
    "domain_orientation_mean",
    "domain_orientation_sd",
    "domain_orientation_min",
    "domain_orientation_max",
    "domain_orientation_range",
    "positive_domain_count",
    "negative_domain_count",
    "neutral_domain_count",
    "domain_consistency_index",
    "orientation_algorithm_version"
]

segment_profile_cols = segment_id_cols + segment_metadata_cols + segment_orientation_cols
segment_profile_cols = [c for c in segment_profile_cols if c in orientation_df.columns]

segment_orientation_profiles = orientation_df[segment_profile_cols].copy()


# ============================================================
# 9. Track-level temporal trajectory 생성
# ============================================================

print("\n6단계: track-level temporal trajectory 생성")

track_col = "parent_track_id_feature" if "parent_track_id_feature" in orientation_df.columns else "parent_track_id_metadata"
segment_order_col = "segment_index_feature" if "segment_index_feature" in orientation_df.columns else "segment_index_metadata"

track_rows = []

for track_id, sub in orientation_df.groupby(track_col):
    sub = sub.copy()
    sub[segment_order_col] = pd.to_numeric(sub[segment_order_col], errors="coerce")
    sub = sub.sort_values(segment_order_col)

    y = pd.to_numeric(sub["overall_acoustic_orientation_index"], errors="coerce").values
    y = y[np.isfinite(y)]

    n = len(y)

    if n == 0:
        continue

    mean_val = float(np.mean(y))
    sd_val = float(np.std(y, ddof=1)) if n > 1 else 0.0
    min_val = float(np.min(y))
    max_val = float(np.max(y))
    range_val = float(max_val - min_val)

    if n > 1:
        slope_val = linear_slope_normalized(y)
        first_last_delta = float(y[-1] - y[0])
        abs_delta_mean = float(np.mean(np.abs(np.diff(y))))
        peak_position = float(np.argmax(y) / (n - 1))
        trough_position = float(np.argmin(y) / (n - 1))
    else:
        slope_val = np.nan
        first_last_delta = np.nan
        abs_delta_mean = np.nan
        peak_position = 0.0
        trough_position = 0.0

    track_rows.append({
        "parent_track_id": track_id,
        "n_segments": n,

        "track_orientation_mean": mean_val,
        "track_orientation_sd": sd_val,
        "track_orientation_min": min_val,
        "track_orientation_max": max_val,
        "track_orientation_range": range_val,
        "track_orientation_slope": slope_val,
        "track_orientation_first_last_delta": first_last_delta,
        "track_orientation_abs_delta_mean": abs_delta_mean,
        "track_orientation_peak_position": peak_position,
        "track_orientation_trough_position": trough_position,

        "instrument_category": safe_unique_join(sub["instrument_category_refined"]) if "instrument_category_refined" in sub.columns else "",
        "primary_instrument": safe_unique_join(sub["primary_instrument_refined"]) if "primary_instrument_refined" in sub.columns else "",
        "genre_level1": safe_unique_join(sub["genre_level1_refined"]) if "genre_level1_refined" in sub.columns else "",
        "genre_level2": safe_unique_join(sub["genre_level2_refined"]) if "genre_level2_refined" in sub.columns else "",
        "vocal_instrumental_type": safe_unique_join(sub["vocal_instrumental_type_refined"]) if "vocal_instrumental_type_refined" in sub.columns else "",
    })

track_trajectory_df = pd.DataFrame(track_rows)

print("track-level 행 수:", len(track_trajectory_df))


# ============================================================
# 10. Track trajectory type 분류
# ============================================================

print("\n7단계: track trajectory type 생성")

# data-adaptive 기준
if len(track_trajectory_df) > 0:
    low_sd_cut = track_trajectory_df["track_orientation_sd"].quantile(0.33)
    high_range_cut = track_trajectory_df["track_orientation_range"].quantile(0.75)
else:
    low_sd_cut = 0.15
    high_range_cut = 0.50

mean_yang_cut = 0.25
mean_yin_cut = -0.25
balanced_mean_cut = 0.20
slope_cut = 0.25

def classify_track_trajectory(row):
    n = row["n_segments"]
    mean_val = row["track_orientation_mean"]
    sd_val = row["track_orientation_sd"]
    slope = row["track_orientation_slope"]
    range_val = row["track_orientation_range"]

    if n == 1:
        if mean_val >= mean_yang_cut:
            return "Single-segment Yang-oriented"
        elif mean_val <= mean_yin_cut:
            return "Single-segment Yin-oriented"
        else:
            return "Single-segment Balanced"

    if mean_val >= mean_yang_cut and sd_val <= low_sd_cut:
        return "Stable-Yang-oriented"

    if mean_val <= mean_yin_cut and sd_val <= low_sd_cut:
        return "Stable-Yin-oriented"

    if abs(mean_val) < balanced_mean_cut and sd_val <= low_sd_cut:
        return "Balanced-Stable"

    if pd.notna(slope) and slope >= slope_cut:
        return "Yin-to-Yang"

    if pd.notna(slope) and slope <= -slope_cut:
        return "Yang-to-Yin"

    if range_val >= high_range_cut:
        return "Dynamic-Mixed"

    return "Mixed-Moderate"

track_trajectory_df["trajectory_type"] = track_trajectory_df.apply(classify_track_trajectory, axis=1)
track_trajectory_df["trajectory_algorithm_version"] = VERSION

trajectory_counts = (
    track_trajectory_df["trajectory_type"]
    .value_counts()
    .reset_index()
)
trajectory_counts.columns = ["trajectory_type", "n"]
trajectory_counts["percent"] = trajectory_counts["n"] / len(track_trajectory_df) * 100

print("\nTrajectory type 분포:")
print(trajectory_counts)


# ============================================================
# 11. Summary tables 생성
# ============================================================

print("\n8단계: summary table 생성")

# domain orientation summary
domain_summary_rows = []

for col in domain_orientation_cols + ["overall_acoustic_orientation_index"]:
    s = pd.to_numeric(orientation_df[col], errors="coerce")
    domain_summary_rows.append({
        "variable": col,
        "n": int(s.notna().sum()),
        "missing": int(s.isna().sum()),
        "mean": float(s.mean()),
        "sd": float(s.std(ddof=1)),
        "min": float(s.min()),
        "p25": float(s.quantile(0.25)),
        "median": float(s.median()),
        "p75": float(s.quantile(0.75)),
        "max": float(s.max())
    })

domain_orientation_summary = pd.DataFrame(domain_summary_rows)

# group별 acoustic profile
group_profile_rows = []

for group, sub in orientation_df.groupby("orientation_profile_group"):
    row = {
        "orientation_profile_group": group,
        "n": len(sub),
        "percent": len(sub) / len(orientation_df) * 100
    }

    for col in domain_orientation_cols + ["overall_acoustic_orientation_index"]:
        row[f"{col}_mean"] = sub[col].mean()
        row[f"{col}_sd"] = sub[col].std(ddof=1)

    group_profile_rows.append(row)

group_profile_summary = pd.DataFrame(group_profile_rows)


# ============================================================
# 12. Validation summary 생성
# ============================================================

print("\n9단계: orientation validation summary 생성")

orientation_validation_summary = pd.DataFrame({
    "item": [
        "input_rows",
        "segment_orientation_rows",
        "track_trajectory_rows",
        "orientation_domains",
        "overall_orientation_missing_rows",
        "orientation_profile_group_missing_rows",
        "yin_oriented_segments",
        "yang_oriented_segments",
        "balanced_segments",
        "mixed_profile_segments",
        "algorithm_version"
    ],
    "value": [
        len(df),
        len(segment_orientation_profiles),
        len(track_trajectory_df),
        len(domain_orientation_cols),
        int(orientation_df["overall_acoustic_orientation_index"].isna().sum()),
        int(orientation_df["orientation_profile_group"].isna().sum()),
        int((orientation_df["orientation_profile_group"] == "Yin-oriented").sum()),
        int((orientation_df["orientation_profile_group"] == "Yang-oriented").sum()),
        int((orientation_df["orientation_profile_group"] == "Balanced").sum()),
        int((orientation_df["orientation_profile_group"] == "Mixed-profile").sum()),
        VERSION
    ]
})


# ============================================================
# 13. Data dictionary 추가 생성
# ============================================================

print("\n10단계: orientation data dictionary 생성")

orientation_column_descriptions = {
    "rhythmic_activity_orientation": "Acoustic orientation index for rhythmic activity based on onset-related features.",
    "dynamic_intensity_orientation": "Acoustic orientation index for dynamic intensity based on RMS energy features.",
    "tempo_activity_orientation": "Acoustic orientation index for temporal activity based on estimated tempo.",
    "spectral_brightness_orientation": "Acoustic orientation index for spectral brightness and timbral sharpness.",
    "tonal_variability_orientation": "Acoustic orientation index for pitch-class variability based on chroma features.",
    "within_segment_dynamics_orientation": "Acoustic orientation index for within-segment temporal variability.",
    "overall_acoustic_orientation_index": "Mean of domain-level acoustic orientation indices. Lower values indicate Yin-oriented acoustic profile and higher values indicate Yang-oriented acoustic profile.",
    "orientation_profile_group": "Rule-based descriptive group derived from overall orientation and domain-level consistency.",
    "domain_orientation_sd": "Standard deviation of domain-level orientation indices.",
    "domain_orientation_range": "Range of domain-level orientation indices.",
    "positive_domain_count": "Number of domains with orientation greater than +0.15.",
    "negative_domain_count": "Number of domains with orientation less than -0.15.",
    "neutral_domain_count": "Number of domains between -0.15 and +0.15.",
    "domain_consistency_index": "Degree of agreement in domain orientation direction.",
    "orientation_algorithm_version": "Version of the acoustic orientation algorithm."
}

orientation_dictionary_rows = []

for col in segment_orientation_profiles.columns:
    orientation_dictionary_rows.append({
        "column_name": col,
        "data_type": str(segment_orientation_profiles[col].dtype),
        "missing_count": int(segment_orientation_profiles[col].isna().sum()),
        "missing_rate_percent": round(segment_orientation_profiles[col].isna().mean() * 100, 4),
        "description": orientation_column_descriptions.get(
            col,
            "Identifier, metadata, or provenance column included in the segment orientation profile table."
        )
    })

orientation_data_dictionary = pd.DataFrame(orientation_dictionary_rows)


# ============================================================
# 14. 그림 저장
# ============================================================

print("\n11단계: figure 저장")

# Figure 1: overall orientation distribution
plt.figure(figsize=(8, 5))
plt.hist(orientation_df["overall_acoustic_orientation_index"].dropna(), bins=50)
plt.xlabel("Overall acoustic orientation index")
plt.ylabel("Number of segments")
plt.title("Distribution of overall acoustic orientation index")
plt.tight_layout()
fig1_path = os.path.join(FIGURE_DIR, "figure_orientation_distribution_v1.png")
plt.savefig(fig1_path, dpi=300)
plt.close()

# Figure 2: orientation group counts
plt.figure(figsize=(8, 5))
plt.bar(group_counts["orientation_profile_group"], group_counts["n"])
plt.xlabel("Orientation profile group")
plt.ylabel("Number of segments")
plt.title("Segment counts by orientation profile group")
plt.xticks(rotation=20)
plt.tight_layout()
fig2_path = os.path.join(FIGURE_DIR, "figure_orientation_group_counts_v1.png")
plt.savefig(fig2_path, dpi=300)
plt.close()

# Figure 3: domain orientation boxplot
plt.figure(figsize=(10, 5))
data_for_boxplot = [orientation_df[col].dropna() for col in domain_orientation_cols]
plt.boxplot(data_for_boxplot, labels=domain_orientation_cols, showfliers=False)
plt.ylabel("Domain orientation index")
plt.title("Domain-level acoustic orientation profiles")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
fig3_path = os.path.join(FIGURE_DIR, "figure_domain_orientation_boxplot_v1.png")
plt.savefig(fig3_path, dpi=300)
plt.close()

# Figure 4: trajectory type counts
if len(trajectory_counts) > 0:
    plt.figure(figsize=(9, 5))
    plt.bar(trajectory_counts["trajectory_type"], trajectory_counts["n"])
    plt.xlabel("Trajectory type")
    plt.ylabel("Number of tracks")
    plt.title("Track-level temporal trajectory types")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    fig4_path = os.path.join(FIGURE_DIR, "figure_track_trajectory_counts_v1.png")
    plt.savefig(fig4_path, dpi=300)
    plt.close()
else:
    fig4_path = ""


# ============================================================
# 15. 파일 저장
# ============================================================

print("\n12단계: CSV / Excel 파일 저장")

# 핵심 파일 경로
segment_orientation_csv = os.path.join(ORIENTATION_DIR, "segment_orientation_profiles_v1.csv")
segment_with_orientation_csv = os.path.join(DATA_DIR, "segment_acoustic_features_with_orientation_v1.csv")
track_trajectory_csv = os.path.join(ORIENTATION_DIR, "track_temporal_trajectory_v1.csv")

feature_stats_csv = os.path.join(VALIDATION_DIR, "orientation_feature_standardization_stats_v1.csv")
domain_scaling_csv = os.path.join(VALIDATION_DIR, "orientation_domain_scaling_stats_v1.csv")
validation_csv = os.path.join(VALIDATION_DIR, "orientation_validation_summary_v1.csv")

domain_summary_csv = os.path.join(TABLE_DIR, "domain_orientation_summary_v1.csv")
group_profile_csv = os.path.join(TABLE_DIR, "orientation_group_profile_summary_v1.csv")
trajectory_counts_csv = os.path.join(TABLE_DIR, "track_trajectory_type_counts_v1.csv")
orientation_dictionary_csv = os.path.join(METADATA_DIR, "data_dictionary_orientation_profiles_v1.csv")

# 저장
segment_orientation_profiles.to_csv(segment_orientation_csv, index=False, encoding="utf-8-sig")
orientation_df.to_csv(segment_with_orientation_csv, index=False, encoding="utf-8-sig")
track_trajectory_df.to_csv(track_trajectory_csv, index=False, encoding="utf-8-sig")

feature_standardization_stats.to_csv(feature_stats_csv, index=False, encoding="utf-8-sig")
domain_scaling_df.to_csv(domain_scaling_csv, index=False, encoding="utf-8-sig")
orientation_validation_summary.to_csv(validation_csv, index=False, encoding="utf-8-sig")

domain_orientation_summary.to_csv(domain_summary_csv, index=False, encoding="utf-8-sig")
group_profile_summary.to_csv(group_profile_csv, index=False, encoding="utf-8-sig")
trajectory_counts.to_csv(trajectory_counts_csv, index=False, encoding="utf-8-sig")
orientation_data_dictionary.to_csv(orientation_dictionary_csv, index=False, encoding="utf-8-sig")


# 통합 Excel
excel_output = os.path.join(OUTPUT_DIR, "gugak_acoustic_orientation_profiles_v1.xlsx")

with pd.ExcelWriter(excel_output, engine="openpyxl") as writer:
    segment_orientation_profiles.to_excel(writer, sheet_name="segment_orientation", index=False)
    track_trajectory_df.to_excel(writer, sheet_name="track_trajectory", index=False)
    mapping_rules_df.to_excel(writer, sheet_name="mapping_rules", index=False)
    domain_orientation_summary.to_excel(writer, sheet_name="domain_summary", index=False)
    group_profile_summary.to_excel(writer, sheet_name="group_profile_summary", index=False)
    trajectory_counts.to_excel(writer, sheet_name="trajectory_counts", index=False)
    orientation_validation_summary.to_excel(writer, sheet_name="validation_summary", index=False)
    orientation_data_dictionary.to_excel(writer, sheet_name="data_dictionary", index=False)


# JSON descriptor
descriptor = {
    "dataset_component": "Acoustic Yin-Yang Orientation Profiles",
    "algorithm_version": VERSION,
    "input_file": INPUT_CSV,
    "n_segments": int(len(orientation_df)),
    "n_tracks": int(len(track_trajectory_df)),
    "orientation_domains": domain_orientation_cols,
    "orientation_value_range": {
        "-1": "Yin-oriented acoustic profile",
        "0": "Balanced or neutral acoustic profile",
        "+1": "Yang-oriented acoustic profile"
    },
    "interpretation_note": (
        "The orientation indices are reproducible acoustic descriptors derived from "
        "rhythmic, dynamic, tempo, spectral, tonal, and within-segment temporal features. "
        "They should not be interpreted as categorical ground-truth labels of Yin or Yang."
    ),
    "generated_files": {
        "segment_orientation_profiles": segment_orientation_csv,
        "segment_acoustic_features_with_orientation": segment_with_orientation_csv,
        "track_temporal_trajectory": track_trajectory_csv,
        "orientation_mapping_rules": mapping_rules_path,
        "validation_summary": validation_csv,
        "excel_workbook": excel_output
    }
}

descriptor_path = os.path.join(ORIENTATION_DIR, "orientation_profile_descriptor_v1.json")

with open(descriptor_path, "w", encoding="utf-8") as f:
    json.dump(descriptor, f, ensure_ascii=False, indent=2)


# ============================================================
# 16. 완료 메시지
# ============================================================

print("\n============================================================")
print("완료: Acoustic Yin–Yang Orientation Profile v1 생성")
print("============================================================")
print("입력 파일:", INPUT_CSV)
print()
print("[핵심 출력 파일]")
print("1) Segment orientation profiles:", segment_orientation_csv)
print("2) Segment full with orientation:", segment_with_orientation_csv)
print("3) Track temporal trajectory:", track_trajectory_csv)
print("4) Orientation mapping rules:", mapping_rules_path)
print("5) Validation summary:", validation_csv)
print("6) 통합 Excel:", excel_output)
print("7) Descriptor JSON:", descriptor_path)
print()
print("[그림]")
print(fig1_path)
print(fig2_path)
print(fig3_path)
print(fig4_path)
print()
print("[Orientation validation summary]")
print(orientation_validation_summary)
print()
print("[Orientation group counts]")
print(group_counts)
print()
print("[Trajectory type counts]")
print(trajectory_counts)

1단계: 병합된 segment dataset 불러오기
행 수: 8928
컬럼 수: 103
확인: 8,928개 segment가 정상적으로 불러와졌습니다.

2단계: domain별 사용 가능한 feature 확인
rhythmic_activity_orientation : ['onset_density', 'onset_count', 'onset_strength_mean', 'onset_strength_std', 'onset_strength_range']
dynamic_intensity_orientation : ['rms_mean', 'rms_std', 'rms_var', 'rms_range', 'rms_cv']
tempo_activity_orientation : ['tempo_bpm']
spectral_brightness_orientation : ['spectral_centroid_mean', 'spectral_centroid_std', 'spectral_bandwidth_mean', 'spectral_rolloff_mean', 'spectral_flatness_mean', 'zero_crossing_rate_mean']
tonal_variability_orientation : ['chroma_std', 'chroma_var', 'chroma_entropy']
within_segment_dynamics_orientation : ['rms_1s_std', 'rms_1s_cv', 'centroid_1s_std', 'centroid_1s_cv', 'onset_strength_1s_std', 'onset_strength_1s_cv']

Mapping rule 저장: D:\gugak_scientific_data_package_v1\orientation_profiles\orientation_mapping_rules_v1.csv

3단계: domain-level orientation 계산 시작
생성된 domain orientation columns:
['rhythmic_activi

In [17]:
 import os
import pandas as pd
import numpy as np

INPUT_CSV = r"D:\gugak_scientific_data_package_v1\data\segment_acoustic_features_with_orientation_v1.csv"

OUTPUT_DIR = r"D:\gugak_scientific_data_package_v1"
DATA_DIR = os.path.join(OUTPUT_DIR, "data")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
VALIDATION_DIR = os.path.join(OUTPUT_DIR, "validation")

df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")

# ------------------------------------------------------------
# 1. 전체 segment 기준 tertile / quintile band 생성
# ------------------------------------------------------------

score_col = "overall_acoustic_orientation_index"

df["orientation_tertile_band"] = pd.qcut(
    df[score_col],
    q=3,
    labels=[
        "Low acoustic orientation",
        "Middle acoustic orientation",
        "High acoustic orientation"
    ],
    duplicates="drop"
)

df["orientation_quintile_band"] = pd.qcut(
    df[score_col],
    q=5,
    labels=[
        "Q1 lowest orientation",
        "Q2 low orientation",
        "Q3 middle orientation",
        "Q4 high orientation",
        "Q5 highest orientation"
    ],
    duplicates="drop"
)

# ------------------------------------------------------------
# 2. 악기 범주별 tertile band도 추가
#    같은 악기군 내 상대적 orientation을 보기 위한 보조 지표
# ------------------------------------------------------------

def within_group_tertile(s):
    try:
        return pd.qcut(
            s,
            q=3,
            labels=[
                "Within-group low",
                "Within-group middle",
                "Within-group high"
            ],
            duplicates="drop"
        )
    except Exception:
        return pd.Series(["Not enough variation"] * len(s), index=s.index)

df["orientation_tertile_within_instrument_category"] = (
    df.groupby("instrument_category_refined")[score_col]
    .transform(lambda x: within_group_tertile(x).astype(str))
)

df["orientation_tertile_within_genre_level1"] = (
    df.groupby("genre_level1_refined")[score_col]
    .transform(lambda x: within_group_tertile(x).astype(str))
)

# ------------------------------------------------------------
# 3. 요약표 생성
# ------------------------------------------------------------

tertile_counts = (
    df["orientation_tertile_band"]
    .value_counts(dropna=False)
    .reset_index()
)
tertile_counts.columns = ["orientation_tertile_band", "n"]
tertile_counts["percent"] = tertile_counts["n"] / len(df) * 100

quintile_counts = (
    df["orientation_quintile_band"]
    .value_counts(dropna=False)
    .reset_index()
)
quintile_counts.columns = ["orientation_quintile_band", "n"]
quintile_counts["percent"] = quintile_counts["n"] / len(df) * 100

profile_by_tertile = (
    df.groupby(["orientation_profile_group", "orientation_tertile_band"])
    .size()
    .reset_index(name="n")
)

instrument_summary = (
    df.groupby("instrument_category_refined")
    .agg(
        n=("overall_acoustic_orientation_index", "count"),
        orientation_mean=("overall_acoustic_orientation_index", "mean"),
        orientation_sd=("overall_acoustic_orientation_index", "std"),
        orientation_median=("overall_acoustic_orientation_index", "median"),
        rhythmic_mean=("rhythmic_activity_orientation", "mean"),
        dynamic_mean=("dynamic_intensity_orientation", "mean"),
        tempo_mean=("tempo_activity_orientation", "mean"),
        spectral_mean=("spectral_brightness_orientation", "mean"),
        tonal_mean=("tonal_variability_orientation", "mean"),
        within_dynamics_mean=("within_segment_dynamics_orientation", "mean")
    )
    .reset_index()
)

genre_summary = (
    df.groupby("genre_level1_refined")
    .agg(
        n=("overall_acoustic_orientation_index", "count"),
        orientation_mean=("overall_acoustic_orientation_index", "mean"),
        orientation_sd=("overall_acoustic_orientation_index", "std"),
        orientation_median=("overall_acoustic_orientation_index", "median"),
        rhythmic_mean=("rhythmic_activity_orientation", "mean"),
        dynamic_mean=("dynamic_intensity_orientation", "mean"),
        tempo_mean=("tempo_activity_orientation", "mean"),
        spectral_mean=("spectral_brightness_orientation", "mean"),
        tonal_mean=("tonal_variability_orientation", "mean"),
        within_dynamics_mean=("within_segment_dynamics_orientation", "mean")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 4. 저장
# ------------------------------------------------------------

output_csv = os.path.join(DATA_DIR, "segment_acoustic_features_with_orientation_v1_plus_bands.csv")
output_xlsx = os.path.join(DATA_DIR, "segment_acoustic_features_with_orientation_v1_plus_bands.xlsx")

df.to_csv(output_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="segment_with_bands", index=False)
    tertile_counts.to_excel(writer, sheet_name="tertile_counts", index=False)
    quintile_counts.to_excel(writer, sheet_name="quintile_counts", index=False)
    profile_by_tertile.to_excel(writer, sheet_name="profile_by_tertile", index=False)
    instrument_summary.to_excel(writer, sheet_name="instrument_summary", index=False)
    genre_summary.to_excel(writer, sheet_name="genre_summary", index=False)

tertile_counts.to_csv(
    os.path.join(TABLE_DIR, "orientation_tertile_counts_v1.csv"),
    index=False,
    encoding="utf-8-sig"
)

quintile_counts.to_csv(
    os.path.join(TABLE_DIR, "orientation_quintile_counts_v1.csv"),
    index=False,
    encoding="utf-8-sig"
)

profile_by_tertile.to_csv(
    os.path.join(TABLE_DIR, "orientation_profile_by_tertile_v1.csv"),
    index=False,
    encoding="utf-8-sig"
)

instrument_summary.to_csv(
    os.path.join(TABLE_DIR, "orientation_summary_by_instrument_category_v1.csv"),
    index=False,
    encoding="utf-8-sig"
)

genre_summary.to_csv(
    os.path.join(TABLE_DIR, "orientation_summary_by_genre_level1_v1.csv"),
    index=False,
    encoding="utf-8-sig"
)

validation = pd.DataFrame({
    "item": [
        "input_rows",
        "output_rows",
        "tertile_band_missing",
        "quintile_band_missing",
        "instrument_within_band_missing",
        "genre_within_band_missing"
    ],
    "value": [
        len(df),
        len(df),
        int(df["orientation_tertile_band"].isna().sum()),
        int(df["orientation_quintile_band"].isna().sum()),
        int(df["orientation_tertile_within_instrument_category"].isna().sum()),
        int(df["orientation_tertile_within_genre_level1"].isna().sum())
    ]
})

validation.to_csv(
    os.path.join(VALIDATION_DIR, "orientation_band_validation_summary_v1.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("완료")
print("저장 CSV:", output_csv)
print("저장 Excel:", output_xlsx)
print()
print("[Tertile counts]")
print(tertile_counts)
print()
print("[Quintile counts]")
print(quintile_counts)

완료
저장 CSV: D:\gugak_scientific_data_package_v1\data\segment_acoustic_features_with_orientation_v1_plus_bands.csv
저장 Excel: D:\gugak_scientific_data_package_v1\data\segment_acoustic_features_with_orientation_v1_plus_bands.xlsx

[Tertile counts]
      orientation_tertile_band     n    percent
0     Low acoustic orientation  2976  33.333333
1  Middle acoustic orientation  2976  33.333333
2    High acoustic orientation  2976  33.333333

[Quintile counts]
  orientation_quintile_band     n   percent
0     Q1 lowest orientation  1786  20.00448
1     Q3 middle orientation  1786  20.00448
2    Q5 highest orientation  1786  20.00448
3        Q2 low orientation  1785  19.99328
4       Q4 high orientation  1785  19.99328


In [18]:
import os
import pandas as pd
import numpy as np

# ============================================================
# 0. 입력 / 출력 경로
# ============================================================

INPUT_CSV = r"D:\gugak_scientific_data_package_v1\data\segment_acoustic_features_with_orientation_v1_plus_bands.csv"

OUTPUT_DIR = r"D:\gugak_scientific_data_package_v1"
DATA_DIR = os.path.join(OUTPUT_DIR, "data")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
ORIENTATION_DIR = os.path.join(OUTPUT_DIR, "orientation_profiles")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(ORIENTATION_DIR, exist_ok=True)

df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")

# ============================================================
# 1. 사용할 orientation domain
# ============================================================

domain_cols = [
    "rhythmic_activity_orientation",
    "dynamic_intensity_orientation",
    "tempo_activity_orientation",
    "spectral_brightness_orientation",
    "tonal_variability_orientation",
    "within_segment_dynamics_orientation"
]

domain_cols = [c for c in domain_cols if c in df.columns]

score_col = "overall_acoustic_orientation_index"

print("사용 domain:", domain_cols)
print("domain 개수:", len(domain_cols))


# ============================================================
# 2. 연속형 방향성 기반 direction label
# ============================================================
# 이건 domain 일치성을 요구하지 않습니다.
# 단순히 전체 acoustic orientation continuum에서 어느 쪽에 가까운지 봅니다.

def classify_direction(x):
    if pd.isna(x):
        return "Missing"
    if x <= -0.25:
        return "Yin-leaning"
    elif x >= 0.25:
        return "Yang-leaning"
    else:
        return "Neutral"

df["orientation_direction_v1b"] = df[score_col].apply(classify_direction)


# ============================================================
# 3. 방향성 강도 label
# ============================================================
# 절댓값이 클수록 음향적으로 한쪽 방향성이 강함

def classify_strength(x):
    if pd.isna(x):
        return "Missing"
    ax = abs(x)
    if ax < 0.15:
        return "Weak orientation"
    elif ax < 0.35:
        return "Moderate orientation"
    else:
        return "Strong orientation"

df["orientation_strength_v1b"] = df[score_col].apply(classify_strength)


# ============================================================
# 4. domain coherence 계산
# ============================================================
# 같은 방향 domain 수와 반대 방향 domain 수를 계산

positive_cut = 0.15
negative_cut = -0.15

df["positive_domain_count_v1b"] = (df[domain_cols] > positive_cut).sum(axis=1)
df["negative_domain_count_v1b"] = (df[domain_cols] < negative_cut).sum(axis=1)
df["neutral_domain_count_v1b"] = len(domain_cols) - (
    df["positive_domain_count_v1b"] + df["negative_domain_count_v1b"]
)

# 주요 방향 domain 수
df["dominant_direction_domain_count_v1b"] = df[
    ["positive_domain_count_v1b", "negative_domain_count_v1b"]
].max(axis=1)

# coherence ratio: 6개 중 몇 개가 한 방향으로 몰렸는지
df["domain_coherence_ratio_v1b"] = (
    df["dominant_direction_domain_count_v1b"] / len(domain_cols)
)

# domain disagreement: 양과 음 domain이 동시에 많은 정도
df["domain_opposition_count_v1b"] = df[
    ["positive_domain_count_v1b", "negative_domain_count_v1b"]
].min(axis=1)


def classify_coherence(row):
    ratio = row["domain_coherence_ratio_v1b"]
    opposition = row["domain_opposition_count_v1b"]

    if ratio >= 0.67 and opposition <= 1:
        return "Coherent"
    elif ratio >= 0.50:
        return "Partially mixed"
    else:
        return "Highly mixed"

df["domain_coherence_group_v1b"] = df.apply(classify_coherence, axis=1)


# ============================================================
# 5. 최종 2차원 profile label
# ============================================================

df["orientation_profile_2d_v1b"] = (
    df["orientation_direction_v1b"] + " | " + df["domain_coherence_group_v1b"]
)

df["orientation_direction_strength_v1b"] = (
    df["orientation_direction_v1b"] + " | " + df["orientation_strength_v1b"]
)


# ============================================================
# 6. 기존 profile group보다 덜 엄격한 relaxed profile도 생성
# ============================================================
# 논문용 주 변수로 쓸 필요는 없지만, 보조 비교용으로 좋습니다.

def classify_relaxed_profile(row):
    x = row[score_col]
    coherence = row["domain_coherence_group_v1b"]

    if pd.isna(x):
        return "Missing"

    if x <= -0.35:
        return "Yin-oriented"
    elif x >= 0.35:
        return "Yang-oriented"
    elif -0.15 < x < 0.15:
        return "Balanced"
    else:
        if x < 0:
            return "Weak Yin-leaning"
        else:
            return "Weak Yang-leaning"

df["orientation_profile_relaxed_v1b"] = df.apply(classify_relaxed_profile, axis=1)


# ============================================================
# 7. 요약표 생성
# ============================================================

def count_table(data, col):
    out = (
        data[col]
        .value_counts(dropna=False)
        .reset_index()
    )
    out.columns = [col, "n"]
    out["percent"] = out["n"] / len(data) * 100
    return out

direction_counts = count_table(df, "orientation_direction_v1b")
strength_counts = count_table(df, "orientation_strength_v1b")
coherence_counts = count_table(df, "domain_coherence_group_v1b")
profile_2d_counts = count_table(df, "orientation_profile_2d_v1b")
relaxed_counts = count_table(df, "orientation_profile_relaxed_v1b")

# 기존 profile과 새 direction 비교
cross_old_new = pd.crosstab(
    df["orientation_profile_group"],
    df["orientation_direction_v1b"],
    margins=True
)

cross_direction_coherence = pd.crosstab(
    df["orientation_direction_v1b"],
    df["domain_coherence_group_v1b"],
    margins=True
)


# ============================================================
# 8. 저장
# ============================================================

output_csv = os.path.join(DATA_DIR, "segment_acoustic_features_with_orientation_v1b_direction_coherence.csv")
output_xlsx = os.path.join(DATA_DIR, "segment_acoustic_features_with_orientation_v1b_direction_coherence.xlsx")

df.to_csv(output_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="segment_v1b", index=False)
    direction_counts.to_excel(writer, sheet_name="direction_counts", index=False)
    strength_counts.to_excel(writer, sheet_name="strength_counts", index=False)
    coherence_counts.to_excel(writer, sheet_name="coherence_counts", index=False)
    profile_2d_counts.to_excel(writer, sheet_name="profile_2d_counts", index=False)
    relaxed_counts.to_excel(writer, sheet_name="relaxed_counts", index=False)
    cross_old_new.to_excel(writer, sheet_name="cross_old_new")
    cross_direction_coherence.to_excel(writer, sheet_name="cross_direction_coherence")

direction_counts.to_csv(
    os.path.join(TABLE_DIR, "orientation_direction_counts_v1b.csv"),
    index=False,
    encoding="utf-8-sig"
)

coherence_counts.to_csv(
    os.path.join(TABLE_DIR, "domain_coherence_counts_v1b.csv"),
    index=False,
    encoding="utf-8-sig"
)

profile_2d_counts.to_csv(
    os.path.join(TABLE_DIR, "orientation_2d_profile_counts_v1b.csv"),
    index=False,
    encoding="utf-8-sig"
)

relaxed_counts.to_csv(
    os.path.join(TABLE_DIR, "orientation_relaxed_profile_counts_v1b.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("완료")
print("저장 CSV:", output_csv)
print("저장 Excel:", output_xlsx)

print("\n[Direction counts]")
print(direction_counts)

print("\n[Coherence counts]")
print(coherence_counts)

print("\n[2D profile counts]")
print(profile_2d_counts)

print("\n[Relaxed profile counts]")
print(relaxed_counts)

print("\n[기존 profile vs 새 direction]")
print(cross_old_new)

print("\n[Direction vs coherence]")
print(cross_direction_coherence)

사용 domain: ['rhythmic_activity_orientation', 'dynamic_intensity_orientation', 'tempo_activity_orientation', 'spectral_brightness_orientation', 'tonal_variability_orientation', 'within_segment_dynamics_orientation']
domain 개수: 6
완료
저장 CSV: D:\gugak_scientific_data_package_v1\data\segment_acoustic_features_with_orientation_v1b_direction_coherence.csv
저장 Excel: D:\gugak_scientific_data_package_v1\data\segment_acoustic_features_with_orientation_v1b_direction_coherence.xlsx

[Direction counts]
  orientation_direction_v1b     n    percent
0                   Neutral  5832  65.322581
1               Yin-leaning  2542  28.472222
2              Yang-leaning   554   6.205197

[Coherence counts]
  domain_coherence_group_v1b     n    percent
0            Partially mixed  6194  69.377240
1               Highly mixed  1966  22.020609
2                   Coherent   768   8.602151

[2D profile counts]
       orientation_profile_2d_v1b     n    percent
0       Neutral | Partially mixed  3886  43.525986

In [26]:
# ============================================================
# 4. Figure 2. 메인 hexbin landscape
# ============================================================

plot_df = df.dropna(
    subset=[
        "overall_acoustic_orientation_index",
        "domain_coherence_ratio_v1b"
    ]
).copy()

# coherence ratio가 이산적이라 시각화용 jitter 추가
rng = np.random.default_rng(20260425)
plot_df["domain_coherence_ratio_plot"] = (
    plot_df["domain_coherence_ratio_v1b"]
    + rng.normal(0, 0.012, len(plot_df))
).clip(0, 1)

X_SPLIT = 0.0
Y_SPLIT = 0.5

def quadrant_label(x, y):
    if x < X_SPLIT and y >= Y_SPLIT:
        return "Yin-side / higher coherence"
    elif x >= X_SPLIT and y >= Y_SPLIT:
        return "Yang-side / higher coherence"
    elif x < X_SPLIT and y < Y_SPLIT:
        return "Yin-side / lower coherence"
    else:
        return "Yang-side / lower coherence"

plot_df["quadrant_label"] = plot_df.apply(
    lambda r: quadrant_label(
        r["overall_acoustic_orientation_index"],
        r["domain_coherence_ratio_v1b"]
    ),
    axis=1
)

quadrant_summary = (
    plot_df["quadrant_label"]
    .value_counts()
    .reset_index()
)
quadrant_summary.columns = ["quadrant", "n"]
quadrant_summary["percent"] = quadrant_summary["n"] / len(plot_df) * 100

q_dict = {
    row["quadrant"]: f"{int(row['n']):,}\n({row['percent']:.1f}%)"
    for _, row in quadrant_summary.iterrows()
}

fig, ax = plt.subplots(figsize=(11, 7))

hb = ax.hexbin(
    plot_df["overall_acoustic_orientation_index"],
    plot_df["domain_coherence_ratio_plot"],
    gridsize=48,
    mincnt=1,
    linewidths=0
)

cbar = fig.colorbar(hb, ax=ax)
cbar.set_label("Number of segments", fontsize=11)

ax.axvline(X_SPLIT, linestyle="--", linewidth=1.6)
ax.axhline(Y_SPLIT, linestyle="--", linewidth=1.6)

ax.set_xlabel("Overall acoustic orientation index", fontsize=12)
ax.set_ylabel("Domain coherence ratio", fontsize=12)

ax.set_title(
    "Figure 2. Distribution of acoustic orientation and domain coherence in Gugak segments",
    fontsize=14,
    fontweight="bold",
    pad=14
)

# ------------------------------------------------------------
# quadrant 설명 박스: 플롯 바깥쪽으로 이동
# ------------------------------------------------------------
box_kw = dict(
    boxstyle="round,pad=0.35",
    facecolor="white",
    edgecolor="black",
    alpha=0.9
)

ax.text(
    -0.24, 0.88,
    "Yin-side\nhigher coherence\n" + q_dict.get("Yin-side / higher coherence", ""),
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=box_kw,
    clip_on=False
)

ax.text(
    0.70, 0.88,
    "Yang-side\nhigher coherence\n" + q_dict.get("Yang-side / higher coherence", ""),
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=box_kw,
    clip_on=False
)

ax.text(
    -0.24, 0.08,
    "Yin-side\nlower coherence\n" + q_dict.get("Yin-side / lower coherence", ""),
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=10,
    bbox=box_kw,
    clip_on=False
)

ax.text(
    0.70, 0.08,
    "Yang-side\nlower coherence\n" + q_dict.get("Yang-side / lower coherence", ""),
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=10,
    bbox=box_kw,
    clip_on=False
)

# 해석 보조 문구
ax.text(
    0.5,
    -0.18,
    "Negative values indicate Yin-side acoustic orientation; positive values indicate Yang-side acoustic orientation.\n"
    "Higher coherence indicates stronger agreement across the six acoustic orientation domains.",
    transform=ax.transAxes,
    ha="center",
    va="top",
    fontsize=10
)

fig2_path = os.path.join(
    FINAL_FIGURE_DIR,
    "Figure_2_acoustic_orientation_coherence_landscape.png"
)

plt.subplots_adjust(left=0.22, right=0.88, bottom=0.18, top=0.88)
plt.savefig(fig2_path, dpi=300, bbox_inches="tight")
plt.close()

In [29]:
# ============================================================
# FINAL CODE
# Figure 1 + Figure 2 + Table 1 생성
# 덮어쓰기 방지: 같은 파일 있으면 _v2, _v3 자동 생성
# ============================================================

from pathlib import Path
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# ============================================================
# 0. 고정 경로
# ============================================================

INPUT_FILE = r"D:\gugak_scientific_data_package_v1\data\segment_acoustic_features_with_orientation_v1b_direction_coherence.csv"

BASE_DIR = r"D:\gugak_scientific_data_package_v1"
FIGURE_DIR = Path(BASE_DIR) / "final_figures"
TABLE_DIR = Path(BASE_DIR) / "final_tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

X_COL = "overall_acoustic_orientation_index"
Y_COL = "domain_coherence_ratio_v1b"
INSTRUMENT_COL = "instrument_category_refined"
GENRE_COL = "genre_level1_refined"

N_SEGMENTS_FOR_FIG1 = 8928


# ============================================================
# 1. 덮어쓰기 방지 함수
# ============================================================

def get_versioned_path(folder, base_name):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)

    candidate = folder / base_name
    if not candidate.exists():
        return candidate

    stem = candidate.stem
    suffix = candidate.suffix

    i = 2
    while True:
        new_candidate = folder / f"{stem}_v{i}{suffix}"
        if not new_candidate.exists():
            return new_candidate
        i += 1


# ============================================================
# 2. 표시용 영문 라벨 생성
# ============================================================

def add_display_labels(df):
    instrument_label_map = {
        "성악": "Vocal",
        "현악기": "Strings",
        "관악기": "Winds",
        "타악기": "Percussion",
        "mixed": "Mixed",
        "unknown": "Unknown"
    }

    genre_label_map = {
        "정악": "Jeongak",
        "민속악": "Folk music",
        "성악": "Vocal music",
        "궁중정재": "Court dance",
        "무속음악": "Shamanic music",
        "타악/연희": "Percussion/Yeonhui",
        "기악": "Instrumental",
        "행악": "Processional music",
        "농악": "Nongak",
        "성악반주": "Vocal accompaniment",
        "unknown": "Unknown"
    }

    df["instrument_display"] = (
        df[INSTRUMENT_COL]
        .astype(str)
        .map(instrument_label_map)
        .fillna(df[INSTRUMENT_COL].astype(str))
    )

    df["genre_display"] = (
        df[GENRE_COL]
        .astype(str)
        .map(genre_label_map)
        .fillna(df[GENRE_COL].astype(str))
    )

    return df


# ============================================================
# 3. Figure 1: Flowchart
# ============================================================

def draw_box(ax, x, y, w, h, text, fontsize=13):
    box = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.02,rounding_size=0.02",
        linewidth=1.4,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(box)
    ax.text(
        x + w / 2,
        y + h / 2,
        text,
        ha="center",
        va="center",
        fontsize=fontsize
    )


def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops=dict(
            arrowstyle="->",
            lw=1.5,
            color="black",
            mutation_scale=18
        )
    )


def make_figure1():
    fig, ax = plt.subplots(figsize=(16, 9), dpi=200)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    ax.text(
        0.5,
        0.95,
        "Figure 1. Dataset construction and acoustic orientation profiling workflow",
        ha="center",
        va="top",
        fontsize=20,
        fontweight="bold"
    )

    box_w = 0.18
    box_h = 0.16

    draw_box(
        ax, 0.05, 0.60, box_w, box_h,
        f"Source audio\nNational Gugak recordings\nSegmented WAV files\nn = {N_SEGMENTS_FOR_FIG1:,}"
    )

    draw_box(
        ax, 0.30, 0.60, box_w, box_h,
        "Metadata annotation\nFolder/file-name rules\nInstrument category\nGenre category\nVocal/instrumental type"
    )

    draw_box(
        ax, 0.55, 0.60, box_w, box_h,
        "Acoustic features\nRhythm\nDynamics\nTempo\nTimbre\nTonal variability"
    )

    draw_box(
        ax, 0.80, 0.60, box_w, box_h,
        "Orientation profiling\n6 acoustic domains\nOverall orientation index\nDomain coherence ratio"
    )

    draw_box(
        ax, 0.30, 0.25, box_w, box_h,
        "Validation records\n0 extraction errors\n100% metadata match\n0 unresolved metadata"
    )

    draw_box(
        ax, 0.55, 0.25, box_w, box_h,
        "Data products\nSegment-level dataset\nTrack-level summary\nMapping rules\nData dictionary"
    )

    draw_arrow(ax, 0.23, 0.68, 0.30, 0.68)
    draw_arrow(ax, 0.48, 0.68, 0.55, 0.68)
    draw_arrow(ax, 0.73, 0.68, 0.80, 0.68)
    draw_arrow(ax, 0.39, 0.60, 0.39, 0.41)
    draw_arrow(ax, 0.64, 0.60, 0.64, 0.41)

    out_path = get_versioned_path(
        FIGURE_DIR,
        "Figure_1_dataset_construction_flowchart.png"
    )

    fig.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)

    return out_path


# ============================================================
# 4. Figure 2: Main landscape
# ============================================================

def make_figure2(df):
    plot_df = df[[X_COL, Y_COL]].dropna().copy()

    plot_df[X_COL] = pd.to_numeric(plot_df[X_COL], errors="coerce")
    plot_df[Y_COL] = pd.to_numeric(plot_df[Y_COL], errors="coerce")
    plot_df = plot_df.dropna().copy()

    total_n = len(plot_df)

    q1 = plot_df[(plot_df[X_COL] < 0) & (plot_df[Y_COL] >= 0.5)]
    q2 = plot_df[(plot_df[X_COL] >= 0) & (plot_df[Y_COL] >= 0.5)]
    q3 = plot_df[(plot_df[X_COL] < 0) & (plot_df[Y_COL] < 0.5)]
    q4 = plot_df[(plot_df[X_COL] >= 0) & (plot_df[Y_COL] < 0.5)]

    n1, n2, n3, n4 = len(q1), len(q2), len(q3), len(q4)

    p1 = 100 * n1 / total_n
    p2 = 100 * n2 / total_n
    p3 = 100 * n3 / total_n
    p4 = 100 * n4 / total_n

    quadrant_df = pd.DataFrame({
        "quadrant": [
            "Yin-side / higher coherence",
            "Yang-side / higher coherence",
            "Yin-side / lower coherence",
            "Yang-side / lower coherence"
        ],
        "n": [n1, n2, n3, n4],
        "percent": [p1, p2, p3, p4]
    })

    fig, ax = plt.subplots(figsize=(12.5, 8.2), dpi=200)

    hb = ax.hexbin(
        plot_df[X_COL],
        plot_df[Y_COL],
        gridsize=38,
        cmap="viridis",
        mincnt=1
    )

    ax.axvline(0, linestyle="--", linewidth=2, color="#1f77b4")
    ax.axhline(0.5, linestyle="--", linewidth=2, color="#1f77b4")

    ax.set_xlim(-0.88, 0.80)
    ax.set_ylim(0.08, 1.06)

    ax.set_xlabel("Overall acoustic orientation index", fontsize=14)
    ax.set_ylabel("Domain coherence ratio", fontsize=14)

    ax.set_title(
        "Figure 2. Distribution of acoustic orientation and domain coherence in Gugak segments",
        fontsize=17,
        fontweight="bold",
        pad=16
    )

    cbar = fig.colorbar(hb, ax=ax)
    cbar.set_label("Number of segments", fontsize=13)

    box_kw = dict(
        boxstyle="round,pad=0.25",
        facecolor="white",
        edgecolor="black",
        alpha=0.88
    )

    ax.text(
        -0.80, 0.88,
        f"Yin-side\nhigher coherence\n{n1:,}\n({p1:.1f}%)",
        ha="left",
        va="center",
        fontsize=12,
        bbox=box_kw
    )

    ax.text(
        0.28, 0.88,
        f"Yang-side\nhigher coherence\n{n2:,}\n({p2:.1f}%)",
        ha="left",
        va="center",
        fontsize=12,
        bbox=box_kw
    )

    ax.text(
        -0.80, 0.22,
        f"Yin-side\nlower coherence\n{n3:,}\n({p3:.1f}%)",
        ha="left",
        va="center",
        fontsize=12,
        bbox=box_kw
    )

    ax.text(
        0.28, 0.22,
        f"Yang-side\nlower coherence\n{n4:,}\n({p4:.1f}%)",
        ha="left",
        va="center",
        fontsize=12,
        bbox=box_kw
    )

    fig.text(
        0.5,
        0.055,
        "Negative values indicate Yin-side acoustic orientation; positive values indicate Yang-side acoustic orientation.\n"
        "Higher coherence indicates stronger agreement across the six acoustic orientation domains.",
        ha="center",
        va="center",
        fontsize=12
    )

    plt.subplots_adjust(left=0.14, right=0.88, top=0.90, bottom=0.18)

    out_path = get_versioned_path(
        FIGURE_DIR,
        "Figure_2_acoustic_orientation_coherence_landscape_adjusted.png"
    )

    fig.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)

    quadrant_path = get_versioned_path(
        TABLE_DIR,
        "Figure_2_quadrant_summary.csv"
    )
    quadrant_df.to_csv(quadrant_path, index=False, encoding="utf-8-sig")

    return out_path, quadrant_df, quadrant_path


# ============================================================
# 5. Table 1
# ============================================================

def format_ci(mean_val, sd_val, n):
    if pd.isna(sd_val) or n <= 1:
        return ""
    se = sd_val / math.sqrt(n)
    low = mean_val - 1.96 * se
    high = mean_val + 1.96 * se
    return f"{low:.3f} to {high:.3f}"


def summarize_group(df, group_type, group_col):
    rows = []

    for group, sub in df.groupby(group_col):
        x = pd.to_numeric(sub[X_COL], errors="coerce").dropna()
        y = pd.to_numeric(sub[Y_COL], errors="coerce").dropna()

        n = len(x)
        if n == 0:
            continue

        mean_x = x.mean()
        sd_x = x.std(ddof=1) if n > 1 else np.nan
        mean_y = y.mean()

        rows.append({
            "Category type": group_type,
            "Group": str(group),
            "n": int(n),
            "Mean orientation index": round(mean_x, 3),
            "SD": round(sd_x, 3) if pd.notna(sd_x) else np.nan,
            "95% CI": format_ci(mean_x, sd_x, n),
            "Mean coherence ratio": round(mean_y, 3)
        })

    out = pd.DataFrame(rows)
    out = out.sort_values("Mean orientation index").reset_index(drop=True)
    return out


def make_table1(df):
    instrument_table = summarize_group(
        df,
        group_type="Instrument",
        group_col="instrument_display"
    )

    genre_table = summarize_group(
        df,
        group_type="Genre",
        group_col="genre_display"
    )

    table1 = pd.concat(
        [instrument_table, genre_table],
        axis=0,
        ignore_index=True
    )

    csv_path = get_versioned_path(
        TABLE_DIR,
        "Table_1_metadata_linked_orientation_summary.csv"
    )

    xlsx_path = get_versioned_path(
        TABLE_DIR,
        "Table_1_metadata_linked_orientation_summary.xlsx"
    )

    table1.to_csv(csv_path, index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        table1.to_excel(writer, sheet_name="Table_1", index=False)
        instrument_table.to_excel(writer, sheet_name="Instrument_summary", index=False)
        genre_table.to_excel(writer, sheet_name="Genre_summary", index=False)

    return table1, csv_path, xlsx_path


# ============================================================
# 6. Supplementary summary
# ============================================================

def make_supplementary(df, quadrant_df):
    direction_counts = (
        df["orientation_direction_v1b"]
        .value_counts()
        .reset_index()
    )
    direction_counts.columns = ["orientation_direction", "n"]
    direction_counts["percent"] = direction_counts["n"] / len(df) * 100

    coherence_counts = (
        df["domain_coherence_group_v1b"]
        .value_counts()
        .reset_index()
    )
    coherence_counts.columns = ["domain_coherence_group", "n"]
    coherence_counts["percent"] = coherence_counts["n"] / len(df) * 100

    summary_df = pd.DataFrame({
        "Metric": [
            "Segment rows",
            "Mean orientation index",
            "SD orientation index",
            "Mean coherence ratio",
            "SD coherence ratio"
        ],
        "Value": [
            len(df),
            round(df[X_COL].mean(), 6),
            round(df[X_COL].std(ddof=1), 6),
            round(df[Y_COL].mean(), 6),
            round(df[Y_COL].std(ddof=1), 6)
        ]
    })

    xlsx_path = get_versioned_path(
        TABLE_DIR,
        "Supplementary_summary_counts.xlsx"
    )

    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        summary_df.to_excel(writer, sheet_name="Summary", index=False)
        direction_counts.to_excel(writer, sheet_name="Direction_counts", index=False)
        coherence_counts.to_excel(writer, sheet_name="Coherence_counts", index=False)
        quadrant_df.to_excel(writer, sheet_name="Quadrant_summary", index=False)

    return xlsx_path


# ============================================================
# 7. 실행
# ============================================================

df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")

required_cols = [
    X_COL,
    Y_COL,
    INSTRUMENT_COL,
    GENRE_COL,
    "orientation_direction_v1b",
    "domain_coherence_group_v1b"
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"필수 컬럼이 없습니다: {missing}")

df = add_display_labels(df)

df[X_COL] = pd.to_numeric(df[X_COL], errors="coerce")
df[Y_COL] = pd.to_numeric(df[Y_COL], errors="coerce")

analysis_df = df.dropna(
    subset=[X_COL, Y_COL, "instrument_display", "genre_display"]
).copy()

fig1_path = make_figure1()
fig2_path, quadrant_df, quadrant_path = make_figure2(analysis_df)
table1_df, table1_csv, table1_xlsx = make_table1(analysis_df)
supp_xlsx = make_supplementary(analysis_df, quadrant_df)

output_log = pd.DataFrame({
    "item": [
        "Figure 1",
        "Figure 2",
        "Table 1 CSV",
        "Table 1 Excel",
        "Supplementary summary",
        "Figure 2 quadrant summary"
    ],
    "path": [
        str(fig1_path),
        str(fig2_path),
        str(table1_csv),
        str(table1_xlsx),
        str(supp_xlsx),
        str(quadrant_path)
    ]
})

log_path = get_versioned_path(TABLE_DIR, "final_output_log.csv")
output_log.to_csv(log_path, index=False, encoding="utf-8-sig")

print("=" * 80)
print("FINAL OUTPUTS GENERATED")
print("=" * 80)
print()
print("[Figures]")
print("Figure 1:", fig1_path)
print("Figure 2:", fig2_path)
print()
print("[Main Table]")
print("Table 1 CSV:", table1_csv)
print("Table 1 Excel:", table1_xlsx)
print()
print("[Supplementary]")
print("Supplementary summary:", supp_xlsx)
print("Quadrant summary:", quadrant_path)
print("Output log:", log_path)
print()
print("[Figure 2 quadrant summary]")
print(quadrant_df)
print()
print("[Table 1 preview]")
print(table1_df)

FINAL OUTPUTS GENERATED

[Figures]
Figure 1: D:\gugak_scientific_data_package_v1\final_figures\Figure_1_dataset_construction_flowchart_v3.png
Figure 2: D:\gugak_scientific_data_package_v1\final_figures\Figure_2_acoustic_orientation_coherence_landscape_adjusted_v2.png

[Main Table]
Table 1 CSV: D:\gugak_scientific_data_package_v1\final_tables\Table_1_metadata_linked_orientation_summary_v3.csv
Table 1 Excel: D:\gugak_scientific_data_package_v1\final_tables\Table_1_metadata_linked_orientation_summary_v3.xlsx

[Supplementary]
Supplementary summary: D:\gugak_scientific_data_package_v1\final_tables\Supplementary_summary_counts_v3.xlsx
Quadrant summary: D:\gugak_scientific_data_package_v1\final_tables\Figure_2_quadrant_summary_v3.csv
Output log: D:\gugak_scientific_data_package_v1\final_tables\final_output_log_v3.csv

[Figure 2 quadrant summary]
                       quadrant     n    percent
0   Yin-side / higher coherence  5065  56.731631
1  Yang-side / higher coherence  1897  21.247760
2 

In [30]:
# ============================================================
# Clean Figure 2 only
# Distribution of acoustic orientation and domain coherence
# - No annotation boxes
# - No explanatory text
# - No overwrite: auto versioned filename
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# 0. 경로 설정
# ============================================================

SEGMENT_CSV = r"D:\gugak_scientific_data_package_v1\data\segment_acoustic_features_with_orientation_v1b_direction_coherence.csv"

OUTPUT_DIR = r"D:\gugak_scientific_data_package_v1"
FINAL_FIGURE_DIR = os.path.join(OUTPUT_DIR, "final_figures")
FINAL_TABLE_DIR = os.path.join(OUTPUT_DIR, "final_tables")

os.makedirs(FINAL_FIGURE_DIR, exist_ok=True)
os.makedirs(FINAL_TABLE_DIR, exist_ok=True)

X_COL = "overall_acoustic_orientation_index"
Y_COL = "domain_coherence_ratio_v1b"

# ============================================================
# 1. 덮어쓰기 방지 함수
# ============================================================

def get_versioned_path(folder, base_name):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)

    candidate = folder / base_name
    if not candidate.exists():
        return candidate

    stem = candidate.stem
    suffix = candidate.suffix

    i = 2
    while True:
        new_candidate = folder / f"{stem}_v{i}{suffix}"
        if not new_candidate.exists():
            return new_candidate
        i += 1

# ============================================================
# 2. 데이터 불러오기
# ============================================================

df = pd.read_csv(SEGMENT_CSV, encoding="utf-8-sig")

df[X_COL] = pd.to_numeric(df[X_COL], errors="coerce")
df[Y_COL] = pd.to_numeric(df[Y_COL], errors="coerce")

plot_df = df.dropna(subset=[X_COL, Y_COL]).copy()

print("Segment rows used:", len(plot_df))

# ============================================================
# 3. Quadrant summary 생성
# ============================================================

X_SPLIT = 0.0
Y_SPLIT = 0.5

def quadrant_label(x, y):
    if x < X_SPLIT and y >= Y_SPLIT:
        return "Yin-side / higher coherence"
    elif x >= X_SPLIT and y >= Y_SPLIT:
        return "Yang-side / higher coherence"
    elif x < X_SPLIT and y < Y_SPLIT:
        return "Yin-side / lower coherence"
    else:
        return "Yang-side / lower coherence"

plot_df["quadrant_label"] = plot_df.apply(
    lambda r: quadrant_label(r[X_COL], r[Y_COL]),
    axis=1
)

quadrant_summary = (
    plot_df["quadrant_label"]
    .value_counts()
    .reset_index()
)

quadrant_summary.columns = ["quadrant", "n"]
quadrant_summary["percent"] = quadrant_summary["n"] / len(plot_df) * 100

quadrant_summary_path = get_versioned_path(
    FINAL_TABLE_DIR,
    "Figure_2_quadrant_summary_clean.csv"
)

quadrant_summary.to_csv(
    quadrant_summary_path,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 4. 시각화용 jitter 추가
# ============================================================

rng = np.random.default_rng(20260425)

plot_df["domain_coherence_ratio_plot"] = (
    plot_df[Y_COL] + rng.normal(0, 0.012, len(plot_df))
).clip(0, 1)

# ============================================================
# 5. Clean Figure 2 생성
# ============================================================

fig, ax = plt.subplots(figsize=(9, 7))

hb = ax.hexbin(
    plot_df[X_COL],
    plot_df["domain_coherence_ratio_plot"],
    gridsize=48,
    mincnt=1,
    linewidths=0,
    cmap="viridis"
)

cbar = fig.colorbar(hb, ax=ax)
cbar.set_label("Number of segments", fontsize=11)

# 기준선만 유지
ax.axvline(X_SPLIT, linestyle="--", linewidth=1.6)
ax.axhline(Y_SPLIT, linestyle="--", linewidth=1.6)

# 축 라벨
ax.set_xlabel("Overall acoustic orientation index", fontsize=12)
ax.set_ylabel("Domain coherence ratio", fontsize=12)

# 제목
ax.set_title(
    "Figure 2. Distribution of acoustic orientation and domain coherence in Gugak segments",
    fontsize=14,
    fontweight="bold",
    pad=14
)

# 축 범위
ax.set_xlim(-0.85, 0.75)
ax.set_ylim(0.08, 1.05)

plt.tight_layout()

fig2_path = get_versioned_path(
    FINAL_FIGURE_DIR,
    "Figure_2_acoustic_orientation_coherence_landscape_clean.png"
)

plt.savefig(fig2_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

# ============================================================
# 6. 완료 출력
# ============================================================

print("=" * 80)
print("Clean Figure 2 generated")
print("=" * 80)
print("Figure 2:", fig2_path)
print("Quadrant summary:", quadrant_summary_path)
print()
print("[Quadrant summary]")
print(quadrant_summary)

Segment rows used: 8928
Clean Figure 2 generated
Figure 2: D:\gugak_scientific_data_package_v1\final_figures\Figure_2_acoustic_orientation_coherence_landscape_clean.png
Quadrant summary: D:\gugak_scientific_data_package_v1\final_tables\Figure_2_quadrant_summary_clean.csv

[Quadrant summary]
                       quadrant     n    percent
0   Yin-side / higher coherence  5065  56.731631
1  Yang-side / higher coherence  1897  21.247760
2    Yin-side / lower coherence  1504  16.845878
3   Yang-side / lower coherence   462   5.174731
